In [ ]:
library(openintro)
library(tidyverse)
library(tidymodels)
library(pscl)
library(forcats)
library(plotly)
library(shiny)
library(cowplot)
library(ggplot2)
library(patchwork)
library(DescTools)
library(boot)
library(readr)
library(here)
library(paletteer)
library(magick)

# Utils

## Tema e Paleta de Cores

In [ ]:
tema <- theme_minimal()
scale_colour <- scale_colour_paletteer_d("ltc::minou")
scale_color <- scale_color_paletteer_d("ltc::minou")
scale_fill <- scale_fill_paletteer_d("ltc::minou")
scale_theme <- paletteer_d("ltc::minou")

scale_theme
homem_cor <- scale_theme[4]
mulher_cor <- scale_theme[5]

## Boxplot - funções

In [ ]:
median_cl_boot <- function(x, conf = 0.95) {
    lconf <- (1 - conf)/2
    uconf <- 1 - lconf
    require(boot)
    bmedian <- function(x, ind) median(x[ind])
    bt <- boot(x, bmedian, 2000)
    bb <- boot.ci(bt, type = "basic")
    data.frame(y = median(x), ymin = quantile(bt$t, lconf), ymax = quantile(bt$t,
        uconf))
}

boxplot_configs <- geom_boxplot(
    # custom boxes
    color="ivory4",
    fill="ivory4",
    alpha=0.2,
    
    outlier.size=3
)

options(warn = -1)
gera_boxplot <- function(dado, marco) {
    g_box <- ggplot(dado, aes(x = situacao_sexo, y = evadiu_periodo, fill = sexo)) +
        geom_boxplot(
            alpha=0.4,
            outlier.size=3
        ) +
        stat_summary(fun.data = median_cl_boot, geom = "errorbar",
        colour = stat_summary_color) + stat_summary(fun.y = median, geom = "point", colour = stat_summary_color) +
        stat_summary(aes(label = round(after_stat(y), 1)), fun.y = median, geom = "text", size = 4, vjust = -0.5, hjust = 2) +
        stat_summary(aes(label = round(after_stat(y), 1)), fun.y = function(x) quantile(x, 0.75), geom = "text", size = 4, vjust = -1, hjust= 2) +
        stat_summary(aes(label = round(after_stat(y), 1)), fun.y = function(x) quantile(x, 0.25), geom = "text", size = 4, vjust = 2, hjust = 2) +
        ylab("Semestre de evasão") +
        xlab("Sexo - tipo de evasão") +
        ggtitle(paste("Semestre de evasão por tipo e sexo -", marco)) +
        theme(
          plot.title=element_text(family='', face='bold', colour='black', size=15),
          panel.grid.major = element_line(colour = "grey"),
          axis.title = element_text(size = rel(1.2))
        ) +
        scale_fill + scale_color +
        tema

    g_box_todos <- ggplot(dado, aes(x = situacao, y = evadiu_periodo)) +
        boxplot_configs +
        stat_summary(fun.data = median_cl_boot, geom = "errorbar",
        colour = stat_summary_color) + stat_summary(fun.y = median, geom = "point", colour = stat_summary_color) +
        stat_summary(aes(label = round(after_stat(y), 1)), fun.y = median, geom = "text", size = 4, vjust = -0.5, hjust = 2) +
        stat_summary(aes(label = round(after_stat(y), 1)), fun.y = function(x) quantile(x, 0.75), geom = "text", size = 4, vjust = -1, hjust= 2) +
        stat_summary(aes(label = round(after_stat(y), 1)), fun.y = function(x) quantile(x, 0.25), geom = "text", size = 4, vjust = 2, hjust = 2) +
        ylab("Semestre de evasão") +
        xlab("Tipo de evasão") +
        ggtitle(paste("Semestre de evasão por tipo -", marco)) +
        theme(
          plot.title=element_text(family='', face='bold', colour='black', size=15),
          panel.grid.major = element_line(colour = "grey"),
          axis.title = element_text(size = rel(1.2))
          ) +
        scale_fill + scale_color +
        tema

    return(list(g_box, g_box_todos))
}

## Bootstrap - Intervalo de confiança - funções

In [ ]:
stat_summary_color <- "firebrick2"

samplemean <- function(dado, indices) {
  return(mean(dado[indices]))
}

ci_para_df <- function(dados_boot, sexo_resultado, situacao_resultado, metrica) {
    conf_level <- 0.95
    ci <- tidy(dados_boot, 
          conf.level = conf_level,
          conf.method = "basic",
          conf.int = TRUE)
    # ci <- boot.ci(dados_boot, conf = 0.95, type = "basic")
    df <- data.frame(
        sexo = sexo_resultado,
        situacao = situacao_resultado,
        conf = conf_level,
        metrica,
        variancia = ci$bias,
        desvio_padra = ci$std.error,
        estatistica = ci$statistic, 
        low = ci$conf.low,
        high = ci$conf.high)
    return(df)
} 


sample.median <- function(dado, indices) {return(median(dado[indices]))}
ci.mean <- function(dado, resample.number,confidence) {
  b <- boot(dado,statistic = samplemean,R = resample.number)
  if(length(b) > 1) {
    temp <- boot.ci(b, conf = confidence, type = "basic")$percent
    if (length(temp) > 0) {
        return(data.frame(mean=mean(dado), low = temp$basic[4], high = temp$basic[5]))
    }
  }
  return(data.frame(mean=mean(dado), low = mean(dado), high = mean(dado)))
}

calcula_bootstrap_desistente_graduado <- function(dado) {
    sexo_filtro <- dado$sexo[1]
    desistentes <- dado %>% filter(desistente == TRUE)
    graduados <- dado %>% filter(graduado == TRUE) %>% mutate(quantidade = n())
    desistentes_b <- boot(data = desistentes$evadiu_periodo,
                       statistic = samplemean, # <- referência para a função
                       R = 2000) # número de bootstraps
    d <- ci_para_df(desistentes_b, sexo_filtro, "Desistente", "media")


    if (!is.na(graduados$quantidade[1])) {
        graduados_b <- boot(data = graduados$evadiu_periodo,
                       statistic = samplemean, # <- referência para a função
                       R = 2000) # número de bootstraps
        g <- ci_para_df(graduados_b, sexo_filtro, "Graduado", "media")
    } else {
        g <- data.frame()
    }

    return(rbind(d, g))
}

calcula_bootstrap_ci <- function(dado, marco_titulo) {
    dado_sexo <- dado %>% mutate(sexo = "Todos")
    g <- calcula_bootstrap_desistente_graduado(dado_sexo)

    dado_sexo <- dado %>% filter(sexo == "Mulher")
    m <- calcula_bootstrap_desistente_graduado(dado_sexo)

    dado_sexo <- dado %>% filter(sexo == "Homem")
    h <- calcula_bootstrap_desistente_graduado(dado_sexo)

    resultado <- rbind(m, h)
    resultado <- rbind(resultado, g) %>% mutate(marco = marco_titulo)

    return(resultado)
}

### Função MODA

getmode <- function(v) {
   uniqv <- unique(v)
   uniqv[which.max(tabulate(match(v, uniqv)))]
}


### Boostrap da proporção de cada status

proporcao_desistentes_boot <- function(x,i) {
  sum(x[i]=="desistente")/length(x)
}

proporcao_graduados_boot <- function(x,i) {
  sum(x[i]=="graduado")/length(x)
}

proporcao_cl_boot <- function(x, conf = 0.95) {
    prop_desistentes <- boot(x, proporcao_desistentes_boot, 2000)
    # https://stackoverflow.com/questions/70948247/interpreting-r-bootstrapping-output-confidence-intervals
    # https://rpubs.com/nazareno/ics-pacote-boot
    desistentes <- ci_para_df(prop_desistentes, "", "Desistentes", "proporção") %>%
        mutate(total_abs = length(x)) %>%
        mutate(status_abs = sum(x == "desistente")) %>%
        rename("status" = situacao)
    prop_graduados <- boot(x, proporcao_graduados_boot, 2000)
    graduados <- ci_para_df(prop_graduados, "", "Graduados", "proporção") %>%
        mutate(total_abs = length(x)) %>%
        mutate(status_abs = sum(x == "graduado")) %>%
        rename("status" = situacao)

    rbind(graduados, desistentes)
}

proporcao_status <- function(x, status) {
  sum(x==status)/length(x)
}

s_d <- function(d, i) {
    agrupado = d %>% 
        slice(i) %>% 
        group_by(sexo) %>% 
        summarise(
            do_grupo = sum(status=="desistente") / n(),
            .groups = "drop")
    homens = agrupado %>% filter(sexo == "Homem") %>% pull(do_grupo)
    mulheres = agrupado %>% filter(sexo == "Mulher") %>% pull(do_grupo)
    homens - mulheres
}

s_g <- function(d, i) {
    agrupado = d %>% 
        slice(i) %>% 
        group_by(sexo) %>% 
        summarise(
            do_grupo = sum(status=="graduado") / n(),
            .groups = "drop")
    homens = agrupado %>% filter(sexo == "Homem") %>% pull(do_grupo)
    mulheres = agrupado %>% filter(sexo == "Mulher") %>% pull(do_grupo)
    homens - mulheres
}

diff_proporcao_status_sexo_cl_boot <- function(x, conf = 0.95) {
    desistentes <- ci_para_df(boot(x, s_d, 2000), "", "Desistentes", "proporção") %>%
        rename("status" = situacao)
    graduados <- ci_para_df(boot(x, s_g, 2000), "", "Graduados", "proporção") %>%
        rename("status" = situacao)
    
    return(rbind(graduados, desistentes))
}

samplemeandiff <- function(d, i) {
    agrupado = d %>% 
        slice(i) %>% 
        group_by(sexo) %>% 
        summarise(
            do_grupo = mean(evadiu_periodo),
            .groups = "drop")
    homens = agrupado %>% filter(sexo == "Homem") %>% pull(do_grupo)
    mulheres = agrupado %>% filter(sexo == "Mulher") %>% pull(do_grupo)
    homens - mulheres
  return(homens-mulheres)
}

diff_semestre_evasao_bootstrap <- function(dado) {
    desistentes <- data.frame()
    graduados <- data.frame()
    
    dado_desistentes <- dado %>% filter(desistente == TRUE)
    if (nrow(dado_desistentes) > 0) {
        desistentes_booted <- boot(dado_desistentes, samplemeandiff, 2000)
        desistentes <- ci_para_df(desistentes_booted, "", "Desistente", "media")
    }

    dado_graduados <- dado %>% filter(graduado == TRUE)
    if (nrow(dado_graduados) > 0) {
        graduados_booted <- boot(dado_graduados, samplemeandiff, 2000)
        graduados <- ci_para_df(graduados_booted, "", "Graduado", "media")    
    }

    return(rbind(desistentes, graduados))
}

## Plota junto - funções

In [ ]:
plota_junto <- function (p1, p2, p3, indice, titulo) {
    combined <- p1[[indice]] + p2[[indice]] +
      plot_annotation(tag_levels = "A") +
      plot_layout(guides = "collect", ncol = 2) &
      theme(plot.tag = element_text(family = "EB Garamond", size = 8), legend.position = "none")

    (combined / p3[[indice]]) +
      plot_annotation(tag_levels = "A", title = titulo) +
      tema +
      theme(
        axis.title = element_text(size = rel(1.2)),
        legend.position = "bottom",
        legend.text.align = 1,
        legend.title = element_text(margin = margin(0, 6, 0, 0)),
        plot.tag = element_text(family = "EB Garamond", size = 6),
        plot.title=element_text(family='', face='bold', colour='black', size=15),
        panel.grid.major = element_line(colour = "grey"),
        text = element_text(margin = margin(0, 3, 0, 2)))
}


plota_junto_2 <- function (p1, p2, indice, titulo) {
    combined <- p1[[indice]] + p2[[indice]] +
        plot_annotation(
            tag_levels = "A",
            title = titulo,
            theme = (tema + theme(legend.position = "bottom"))
        ) +
        plot_layout(guides = "collect")
}

plota_semestre_evasao_marco <- function(dado, marco_titulo) {
    evasao <- dado %>%
        filter(situacao == "Desistente") %>%
        ggplot() +
        geom_linerange(aes(x=sexo, ymin=low, ymax=high), colour="orange", alpha=0.9, size=5) +
        geom_point(aes(x=sexo, y=estatistica), colour="blue", alpha=0.9, size=1.3) +
        coord_flip() +
        ggtitle(paste("Semestre de evasão (IC) - Desistentes -", marco_titulo)) +
        xlab("Sexo") +
        ylab("Semestre") +
        scale_fill + scale_color +
        tema +
        theme(
          plot.title=element_text(family='', face='bold', colour='black', size=15),
          panel.grid.major = element_line(colour = "grey"),
          axis.title = element_text(size = rel(1.2)))

    tem_concluintes <- nrow(dado %>%
        filter(situacao == "Graduado")) > 0

    conclusao <- NA
    if (tem_concluintes) {
       conclusao <- dado %>%
            filter(situacao == "Graduado") %>%
            ggplot() +
            geom_linerange(aes(x=sexo, ymin=low, ymax=high), colour="orange", alpha=0.9, size=5) +
            geom_point(aes(x=sexo, y=estatistica), colour="blue", alpha=0.9, size=1.3) +
            coord_flip() +
            ggtitle(paste("Semestre de evasão (IC) - Concluintes -", marco_titulo)) +
            xlab("Sexo") +
            ylab("Semestre") +
            scale_fill + scale_color +
            tema +
            theme(
              plot.title=element_text(family='', face='bold', colour='black', size=15),
              panel.grid.major = element_line(colour = "grey"),
              axis.title = element_text(size = rel(1.2)))
    }


    return(list(evasao, conclusao))
}

# Preparando os Dados

In [ ]:
var_menor_ano = 2006
var_maior_ano = 2020
var_maior_periodo = sprintf("%d.%d", var_maior_ano, 2)
var_ano_limite = 2017
var_periodo_limite = sprintf("%d.%d", var_ano_limite, 2)

alunos = read_csv(here::here("/input/alunos_computacao.csv"), show_col_types = FALSE)
historico = read_csv(here::here("/input/historico_computacao_notas.csv"), show_col_types = FALSE)

historico_cra <- historico %>%
  reframe(aluno_matricula, cra) %>%
  rename(matricula = aluno_matricula) %>%
  filter(!duplicated(matricula))

alunos <- merge(x=alunos, y=historico_cra, by="matricula")

glimpse(alunos %>% arrange(curriculo))

alunos %>%
  reframe(forma_evasao) %>%
  filter(!duplicated(forma_evasao))

In [ ]:
alunos <- alunos %>%
  mutate(graduado = ifelse((forma_evasao == "GRADUADO" | forma_evasao == "CONCLUIDO - NAO COLOU GRAU"), TRUE, FALSE)) %>%
  mutate(regular = ifelse((forma_evasao == "REGULAR"), TRUE, FALSE)) %>%
  mutate(desistente = ifelse((graduado != TRUE & regular != TRUE), TRUE, FALSE))

alunos <- alunos %>%
  filter(periodo_ingresso_concat >= 2006) %>%
  arrange(periodo_ingresso_concat) %>%
  group_by(cpf) %>%
  mutate(reingressos_numero = n() - 1) %>%
  mutate(evadiu_periodo = ifelse(is.na(last(evadiu_periodo)), NA, sum(evadiu_periodo) - reingressos_numero)) %>%
  mutate(periodo_ingresso_concat = min(periodo_ingresso_concat)) %>%
  mutate(periodo_evasao_concat = ifelse(is.na(evadiu_periodo), NA, max(periodo_evasao_concat))) %>%
  mutate(curriculo = max(curriculo)) %>%
  mutate(forma_ingresso = first(forma_ingresso)) %>%
  mutate(forma_evasao = last(forma_evasao)) %>%
  mutate(reserva_vagas = first(reserva_vagas)) %>%
  mutate(media_vestibular = first(media_vestibular)) %>%
  mutate(idade_ingresso = min(idade_ingresso)) %>%
  mutate(idade_evasao = max(idade_evasao)) %>%
  mutate(desistente = last(desistente)) %>%
  mutate(regular = last(regular)) %>%
  mutate(graduado = last(graduado)) %>%
  mutate(cra = signif(last(cra), digits = 2)) %>%
  ungroup() %>%
  filter(!duplicated(cpf)) %>%
  select(curriculo, forma_ingresso, forma_evasao, reserva_vagas, cpf, sexo,
         ano_nascimento, cor, tipo_ensino_medio, media_vestibular, idade_ingresso,
         idade_evasao, periodo_ingresso_concat, periodo_evasao_concat, evadiu_periodo,
         reingressos_numero, graduado, regular, desistente, cra)

alunos <- alunos %>%
  mutate(eficiencia = 0) %>%
  mutate(eficiencia = ifelse(curriculo == 1999,
                             ifelse(graduado,
                                    ifelse(evadiu_periodo <= 8, 1, 0),
                                    0),
                             ifelse(graduado,
                                    ifelse(evadiu_periodo <= 9, 1, 0),
                                    0)
                             ))

alunos <- alunos %>%
  mutate(sucesso = 0) %>%
  mutate(sucesso = ifelse(curriculo == 1999,
                             ifelse(graduado,
                                    ifelse(evadiu_periodo <= 12, 1, 0),
                                    0),
                             ifelse(graduado,
                                    ifelse(evadiu_periodo <= 14, 1, 0),
                                    0)
                             ))

alunos <- alunos %>%
   mutate(concluinte = graduado) %>%
   mutate(desistiu = desistente) %>%
   mutate(cursante = regular) %>%
   gather(key="status", value="estado", 17:19) %>%
   filter(estado == TRUE) %>%
   select(-estado) %>%
   rename(graduado = concluinte) %>%
   rename(desistente = desistiu) %>%
   rename(regular = cursante)


glimpse(alunos)

In [ ]:
calcula_propocao_mulheres <- function(dados) {
    total_mulheres = (dados %>% filter(sexo == "Mulher"))$total_ingressos[1]
    print(total_mulheres)
    total_todos = (dados %>% filter(sexo == "Todos"))$total_ingressos[1]
    print(total_todos)
    proporcao_mulheres = 100 * (total_mulheres/total_todos)
    dados_com_proporcao <- dados
    dados_com_proporcao["proporcao_mulheres"] = proporcao_mulheres

    return(dados_com_proporcao)
}

calcula_sexo_todos <- function (dados) {
    todos <- dados %>%
        group_by(evadiu_periodo) %>%
        summarise(
            sexo = "Todos",
            total_ingressos = sum(total_ingressos),
            evadiu_periodo = first(evadiu_periodo),
            regulares = sum(regulares),
            graduados = sum(graduados),
            graduados_acumulado = sum(graduados_acumulado),
            desistentes = sum(desistentes),
            desistentes_acumulado = sum(desistentes_acumulado),
            sucesso = sum(sucesso),
            eficiencia = sum(eficiencia))

    alunos_gerais <- rbind(dados, todos)
}

calcula_taxas_base <- function (dados) {
    # por periodo
    resultado <- dados %>%
        group_by(sexo) %>%
        mutate(total_ingressos = n()) %>%
        mutate(sucesso = sum(sucesso)) %>%
        mutate(eficiencia = sum(eficiencia)) %>%
        filter(evadiu_periodo >= 0) %>%
        group_by(sexo, graduado, evadiu_periodo) %>%
        mutate(graduados = ifelse(graduado == TRUE, n(), 0)) %>%
        group_by(sexo, desistente, evadiu_periodo) %>%
        mutate(desistentes = ifelse(desistente == TRUE, n(), 0)) %>%
        group_by(sexo, evadiu_periodo) %>%
        mutate(graduados = max(graduados)) %>%
        mutate(desistentes = max(desistentes)) %>%
        filter(!duplicated(cbind(evadiu_periodo, graduados, desistentes))) %>%
        ungroup() %>%
        reframe(sexo, total_ingressos, evadiu_periodo, graduados, desistentes, sucesso, eficiencia) %>%
        complete(sexo, evadiu_periodo) %>%
        replace(is.na(.), 0) %>%
        group_by(sexo) %>%
        mutate(total_ingressos = max(total_ingressos)) %>%
        mutate(sucesso = max(sucesso)) %>%
        mutate(eficiencia = max(eficiencia))

    # agrupado
    resultado <- resultado %>%
        group_by(sexo) %>%
        arrange(evadiu_periodo) %>%
        mutate(graduados_acumulado = cumsum(graduados)) %>%
        mutate(desistentes_acumulado = cumsum(desistentes)) %>%
        mutate(regulares = total_ingressos - (graduados_acumulado + desistentes_acumulado)) %>%
        ungroup() %>%
        filter(!duplicated(cbind(sexo, evadiu_periodo))) %>%
        reframe(sexo, total_ingressos, evadiu_periodo, regulares, graduados, graduados_acumulado, desistentes, desistentes_acumulado, sucesso, eficiencia) %>%
        arrange(sexo)

    resultado <- calcula_sexo_todos(resultado)
    resultado <- calcula_propocao_mulheres(resultado)

    return(resultado)
}

calcula_taxas_inep <- function(dados) {
  taxas_inep <- dados %>%
    group_by(sexo) %>%
    mutate(taxa_permanencia = (regulares/total_ingressos) * 100) %>%
    mutate(taxa_conclusao_acumulada = (graduados_acumulado/total_ingressos) * 100) %>%
    mutate(taxa_conclusao_periodo = (graduados/total_ingressos) * 100) %>%
    mutate(taxa_desistencia_acumulada = (desistentes_acumulado/total_ingressos) * 100) %>%
    mutate(taxa_desistencia_periodo = (desistentes/total_ingressos) * 100) %>%
    mutate(taxa_maxima_sucesso = ((regulares + graduados_acumulado) / total_ingressos) * 100) %>%
    mutate(tempo_total_conclusao = evadiu_periodo * graduados) %>%
    mutate(tempo_total_conclusao = sum(tempo_total_conclusao)) %>%
    mutate(tempo_medio_conclusao = tempo_total_conclusao/max(graduados_acumulado)) %>%
    ungroup() %>%
    mutate(taxa_eficiencia = (eficiencia / total_ingressos) * 100) %>%
    mutate(taxa_sucesso = (sucesso / total_ingressos) * 100) %>%
    mutate(taxa_sucesso_final = (sucesso / total_ingressos) * 100) %>%
    mutate(pcp = (eficiencia/sucesso) * 100) %>%
    mutate(taxa_insucesso = ((total_ingressos - sucesso)/total_ingressos) * 100) %>%
    filter(!duplicated(cbind(evadiu_periodo, sexo)))

  return(taxas_inep)
}

______________________________________________________________

# Análise Exploratória dos Dados

In [ ]:
analise_exp_1 <- alunos %>%
    mutate(periodo_ingresso_concat = as.character(periodo_ingresso_concat)) %>%
    ggplot(aes(y = sort(periodo_ingresso_concat), fill = sexo)) +
    geom_bar() +
    scale_fill + scale_color +
    tema +
    xlab("Número de ingressos") +
    ylab("Período de ingresso") +
    ggtitle("Número de ingressos por sexo")

analise_exp_2 <- alunos %>%
    mutate(periodo_ingresso_concat = as.character(periodo_ingresso_concat)) %>%
    ggplot(aes(y = sort(periodo_ingresso_concat), fill = sexo)) +
    geom_bar(position="fill") +
    scale_fill + scale_color +
    tema +
    xlab("Proporção") +
    ylab("Período de ingresso") +
    ggtitle("Proporção de sexo dos ingressantes")

analise_exp_3 <- alunos %>%
    ggplot(aes(x = reingressos_numero)) +
    geom_bar() +
    scale_fill + scale_color +
    tema +
    scale_x_discrete(name ="Número de reingressos por pessoa", 
                    limits=c(0,1,2,3,4,5,6)) +
    scale_y_continuous(name = "Número de pessoas",
                       trans='log10') +
    ggtitle("Número de reingresso por pessoa")

analise_exp_4 <- alunos %>%
    mutate(reingressos_numero = as.character(reingressos_numero)) %>%
    ggplot(aes(x = status, fill = reingressos_numero)) +
    geom_bar(position="fill") +
    scale_fill + scale_color +
    tema +
    xlab("Status no curso") +
    ylab("Proporção") +
    labs(fill="Número de vezes que reingressou no curso") +
    ggtitle("Proporção de reingressos por graduados, \ndesistentes e regulares no curso")

analise_exp_5 <- alunos %>%
    ggplot(aes(x = fct_infreq(cor))) +
    geom_bar() +
    scale_fill + scale_color +
    tema +
    xlab("Cor do aluno") +
    ylab("Número de alunos") +
    ggtitle("Cor dos alunos")

analise_exp_6 <- alunos %>%
    mutate(periodo_ingresso_concat = as.character(periodo_ingresso_concat)) %>%
    ggplot(aes(y = sort(periodo_ingresso_concat), fill = cor)) +
    geom_bar(position="fill") +
    scale_fill + scale_color +
    tema +
    xlab("Proporção de cor") +
    ylab("Período de ingresso") +
    ggtitle("Proporção de cores dos alunos por período de ingresso")

analise_exp_7 <- alunos %>%
    ggplot(aes(y = cor, fill = sexo)) +
    geom_bar(position="fill") +
    scale_fill + scale_color +
    tema +
    xlab("Proporção de sexo") +
    ylab("Cor") +
    ggtitle("Proporção de de sexo por cor")

analise_exp_8 <- alunos %>%
    mutate(periodo_ingresso_concat = as.character(periodo_ingresso_concat)) %>%
    mutate(reingressos_numero = as.character(reingressos_numero)) %>%
    ggplot(aes(y = periodo_ingresso_concat, fill = reingressos_numero)) +
    geom_bar(position="fill") +
    scale_fill + scale_color +
    tema +
    xlab("Proporção de reingressos") +
    ylab("Período de ingresso") +
    labs(fill="Número de vezes que reingressou no curso") +
    ggtitle("Proporção de reingressos por período de ingresso")

analise_exp_9 <- alunos %>%
    mutate(periodo_ingresso_concat = as.character(periodo_ingresso_concat)) %>%
    mutate(status = as.character(status)) %>%
    ggplot(aes(y = periodo_ingresso_concat, fill = status)) +
    geom_bar(position="fill") +
    scale_fill + scale_color +
    tema +
    xlab("Proporção de alunos por status") +
    ylab("Período de ingresso") +
    ggtitle("Proporção de alunos por status por período de ingresso")

analise_exp_1
analise_exp_2
analise_exp_3
analise_exp_4
analise_exp_5
analise_exp_6
analise_exp_7
analise_exp_8
analise_exp_9

--------------------------

In [ ]:
filtra_alunos_por_ingresso <- function (alunos, primeiro_periodo, ultimo_periodo) {
    return (
        alunos %>%
            filter(periodo_ingresso_concat >= primeiro_periodo) %>%
            filter(periodo_ingresso_concat <= ultimo_periodo) %>%
            group_by(periodo_ingresso_concat) %>%
            mutate(ingressos_periodo = n()) %>%
            mutate(desistentes = sum(!regular)) %>%
            mutate_at(vars(evadiu_periodo), ~replace_na(., 0)) %>%
            mutate(maior_periodo_evasao = max(evadiu_periodo)) %>%
            ungroup() %>%
            mutate(proporcao_desistentes = 100*desistentes/ingressos_periodo) %>%
            filter(!duplicated(periodo_ingresso_concat)) %>%
            arrange(periodo_ingresso_concat) %>%
            mutate(periodo = row_number()) %>%
            reframe(
                periodo_ingresso_concat,
                ingressos_periodo,
                desistentes,
                proporcao_desistentes,
                periodo,
                maior_periodo_evasao
            )
    )
}

# Agregado (2006-2014)

In [ ]:
primeiro_periodo = 2006.1
ultimo_periodo = 2014.2

marco_titulo <- paste("(", primeiro_periodo, " - ", ultimo_periodo, ")")

aux <- filtra_alunos_por_ingresso(alunos, primeiro_periodo, ultimo_periodo)

ultimo_periodo_todos = max(aux$periodo_ingresso_concat)
total_periodos = min(aux$maior_periodo_evasao)

print(sprintf("%s: %1.1f", "Último período a ser analisado nos dados gerais", ultimo_periodo_todos))
print(sprintf("%s: %i", "Total de períodos analisados", total_periodos))

alunos_agregados_raw <- alunos %>%
    mutate_at(vars(evadiu_periodo), ~replace_na(., 0)) %>%
    filter(periodo_ingresso_concat <= ultimo_periodo_todos) %>%
    filter(evadiu_periodo <= total_periodos)

alunos_agregados_taxas <- calcula_taxas_base(alunos_agregados_raw)

alunos_agregados_raw["proporcao_mulheres"] = alunos_agregados_taxas$proporcao_mulheres[1]
alunos_agregados_taxas <- calcula_taxas_inep(alunos_agregados_taxas)

dado_raw <- alunos_agregados_raw
dado_taxas <- alunos_agregados_taxas

dado_bp <- dado_raw %>%
    filter(regular == FALSE) %>%
    mutate(situacao = ifelse(graduado == TRUE, "graduado", "desistente"))%>%
    mutate(situacao_sexo = paste(str_to_title(situacao), sexo))

boxplots <- gera_boxplot(dado_bp, marco_titulo)
boxplot_agregado <- boxplots
print(boxplots[[1]])
print(boxplots[[2]])

resultado <- calcula_bootstrap_ci(dado_raw, marco_titulo) %>% mutate(marco = marco_titulo)
resultado

diff_semestre_evasao_df <- diff_semestre_evasao_bootstrap(dado_raw) %>% mutate(marco = marco_titulo)
diff_semestre_evasao_df

graficos <- plota_semestre_evasao_marco(resultado, marco_titulo)
print(graficos[[1]])
print(graficos[[2]])


## Tabela ICs

In [ ]:
homens <- dado_raw %>%
  filter(sexo == "Homem")

mulheres <- dado_raw %>%
  filter(sexo == "Mulher")

ci_homens <- proporcao_cl_boot(homens$status) %>%
  mutate(sexo = "Homens")

ci_mulheres <- proporcao_cl_boot(mulheres$status) %>%
  mutate(sexo = "Mulheres")

evasao_ic_agregado <- rbind(ci_homens, ci_mulheres)
evasao_ic_agregado

In [ ]:
grafico_evasao_ic <- function(dado, marco) {
    return(
        dado %>%
        ggplot() +
        geom_linerange(aes(x = status, y = estatistica, ymin = low, ymax = high, color = sexo, fill = sexo), alpha = 0.3, size = 20) +
        geom_point(aes(x = status, y = estatistica, color = sexo, fill = sexo), size = 3) +
        xlab("Status") +
        ylab("Proporção de alunos") +
        ggtitle(sprintf("Proporção de alunos por tipo de evasão %s", marco))+
        scale_fill + scale_color +
        tema +
        theme(
          plot.title=element_text(family='', face='bold', colour='black', size=15),
          panel.grid.major = element_line(colour = "grey"),
          axis.title = element_text(size = rel(1.2))
        )
    )
}

grafico_evasao_ic(evasao_ic_agregado, "(2006 - 2014)")

In [ ]:
semestres <- 1:12

evasao_ic_agregado <- data.frame()

for(semestre in semestres) {
    status_por_periodo <- dado_raw %>%
      mutate(status = ifelse(evadiu_periodo == semestre, status, "outro"))

    ci_status <- diff_proporcao_status_sexo_cl_boot(status_por_periodo) %>%
      mutate(semestre_evasao = semestre)

    evasao_ic_agregado <- rbind(evasao_ic_agregado, ci_status)
}

evasao_ic_agregado

In [ ]:
semestres <- 1:12

evasao_ic_agregado <- data.frame()

for(semestre in semestres) {
    homens <- dado_raw %>%
      filter(sexo == "Homem") %>%
      mutate(status = ifelse(evadiu_periodo == semestre, status, "outro"))

    mulheres <- dado_raw %>%
      filter(sexo == "Mulher") %>%
      mutate(status = ifelse(evadiu_periodo == semestre, status, "outro"))


    ci_homens <- proporcao_cl_boot(homens$status) %>%
      mutate(sexo = "Homens") %>%
      mutate(semestre_evasao = semestre)

    ci_mulheres <- proporcao_cl_boot(mulheres$status) %>%
      mutate(sexo = "Mulheres") %>%
      mutate(semestre_evasao = semestre)

    evasao_ic_agregado_aux <- rbind(ci_homens, ci_mulheres)
    evasao_ic_agregado <- rbind(evasao_ic_agregado, evasao_ic_agregado_aux)
}

evasao_ic_agregado

In [ ]:
ic_evasao_agregado_sexo <- evasao_ic_agregado %>%
  filter(status == "Desistentes") %>%
  ggplot() +
  geom_linerange(aes(x = semestre_evasao, y = estatistica, ymin = low, ymax = high, color = sexo, fill = sexo), alpha = 0.3, size = 10) +
  geom_point(aes(x = semestre_evasao, y = estatistica, color = sexo, fill = sexo), size = 3) +
  xlab("Semestre") +
  ylab("Proporção de alunos") +
  ggtitle("Intervalo de confiança da média da proporção\n de alunos desistentes por semestre")+
  scale_fill + scale_color +
  tema +
  theme(
    plot.title=element_text(family='', face='bold', colour='black', size=15),
    panel.grid.major = element_line(colour = "grey"),
    axis.title = element_text(size = rel(1.2))
  )

ic_evasao_agregado_sexo

In [ ]:
ic_conclusao_agregado_sexo <- evasao_ic_agregado %>%
  filter(status == "Graduados") %>%
  ggplot() +
  geom_linerange(aes(x = semestre_evasao, y = estatistica, ymin = low, ymax = high, color = sexo, fill = sexo), alpha = 0.3, size = 10) +
  geom_point(aes(x = semestre_evasao, y = estatistica, color = sexo, fill = sexo), size = 3) +
  xlab("Semestre") +
  ylab("Proporção de alunos") +
  ggtitle("Intervalo de confiança da média da proporção\n de alunos concluintes por semestre")+
  scale_fill + scale_color +
  tema +
  theme(
    plot.title=element_text(family='', face='bold', colour='black', size=15),
    panel.grid.major = element_line(colour = "grey"),
    axis.title = element_text(size = rel(1.2))
  )
ic_conclusao_agregado_sexo

--------------------------

# Marco 1 (2006-2008)

In [ ]:
primeiro_periodo = 2006.1
ultimo_periodo = 2008.2

marco_titulo <- paste("(", primeiro_periodo, " - ", ultimo_periodo, ")")

aux <- filtra_alunos_por_ingresso(alunos, primeiro_periodo, ultimo_periodo)

total_periodos = min(aux$maior_periodo_evasao)

print(sprintf("%s: %i", "Total de períodos analisados", total_periodos))

alunos_marco_1_raw <- alunos %>%
    mutate_at(vars(evadiu_periodo), ~replace_na(., 0)) %>%
    filter(periodo_ingresso_concat >= primeiro_periodo) %>%
    filter(periodo_ingresso_concat <= ultimo_periodo) %>%
    filter(evadiu_periodo <= total_periodos)

alunos_marco_1_taxas <- calcula_taxas_base(alunos_marco_1_raw)
alunos_marco_1_taxas["proporcao_mulheres"] = alunos_marco_1_taxas$proporcao_mulheres[1]
alunos_marco_1_taxas <- calcula_taxas_inep(alunos_marco_1_taxas)

dado_raw <- alunos_marco_1_raw
dado_taxas <- alunos_marco_1_taxas

dado_bp <- dado_raw %>%
    filter(regular == FALSE) %>%
    mutate(situacao = ifelse(graduado == TRUE, "graduado", "desistente"))%>%
    mutate(situacao_sexo = paste(str_to_title(situacao), sexo))

boxplots <- gera_boxplot(dado_bp, marco_titulo)
print(boxplots[[1]])
print(boxplots[[2]])

resultado_novo <- calcula_bootstrap_ci(dado_raw, marco_titulo) %>% mutate(marco = marco_titulo)
resultado <- rbind(resultado, resultado_novo)
resultado

diff_semestre_evasao_df_novo <- diff_semestre_evasao_bootstrap(dado_raw) %>% mutate(marco = marco_titulo)
diff_semestre_evasao_df <- rbind(diff_semestre_evasao_df, diff_semestre_evasao_df_novo)
diff_semestre_evasao_df

In [ ]:
homens <- dado_raw %>%
  filter(sexo == "Homem")

mulheres <- dado_raw %>%
  filter(sexo == "Mulher")

ci_homens <- proporcao_cl_boot(homens$status) %>%
  mutate(sexo = "Homens")

ci_mulheres <- proporcao_cl_boot(mulheres$status) %>%
  mutate(sexo = "Mulheres")

evasao_ic_m1 <- rbind(ci_homens, ci_mulheres)

grafico_evasao_ic(evasao_ic_m1, "(2006-2004)")

----------------------

# Marco 2 (2009-2012)

In [ ]:
primeiro_periodo = 2009.1
ultimo_periodo = 2012.2

marco_titulo <- paste("(", primeiro_periodo, " - ", ultimo_periodo, ")")

aux <- filtra_alunos_por_ingresso(alunos, primeiro_periodo, ultimo_periodo)

total_periodos = min(aux$maior_periodo_evasao)

print(sprintf("%s: %i", "Total de períodos analisados", total_periodos))

alunos_marco_2_raw <- alunos %>%
    mutate_at(vars(evadiu_periodo), ~replace_na(., 0)) %>%
    filter(periodo_ingresso_concat >= primeiro_periodo) %>%
    filter(periodo_ingresso_concat <= ultimo_periodo) %>%
    filter(evadiu_periodo <= total_periodos)

alunos_marco_2_taxas <- calcula_taxas_base(alunos_marco_2_raw)
alunos_marco_2_taxas["proporcao_mulheres"] = alunos_marco_2_taxas$proporcao_mulheres[1]
alunos_marco_2_taxas <- calcula_taxas_inep(alunos_marco_2_taxas)

dado_raw <- alunos_marco_2_raw
dado_taxas <- alunos_marco_2_taxas

dado_bp <- dado_raw %>%
    filter(regular == FALSE) %>%
    mutate(situacao = ifelse(graduado == TRUE, "graduado", "desistente"))%>%
    mutate(situacao_sexo = paste(str_to_title(situacao), sexo))

boxplots <- gera_boxplot(dado_bp, marco_titulo)
print(boxplots[[1]])
print(boxplots[[2]])

resultado_novo <- calcula_bootstrap_ci(dado_raw, marco_titulo) %>% mutate(marco = marco_titulo)
resultado <- rbind(resultado, resultado_novo)
resultado

diff_semestre_evasao_df_novo <- diff_semestre_evasao_bootstrap(dado_raw) %>% mutate(marco = marco_titulo)
diff_semestre_evasao_df <- rbind(diff_semestre_evasao_df, diff_semestre_evasao_df_novo)
diff_semestre_evasao_df

In [ ]:
homens <- dado_raw %>%
  filter(sexo == "Homem")

mulheres <- dado_raw %>%
  filter(sexo == "Mulher")

ci_homens <- proporcao_cl_boot(homens$status) %>%
  mutate(sexo = "Homens")

ci_mulheres <- proporcao_cl_boot(mulheres$status) %>%
  mutate(sexo = "Mulheres")

evasao_ic_m2 <- rbind(ci_homens, ci_mulheres)

grafico_evasao_ic(evasao_ic_m2, "(2009-2012)")

----------------------

# Marco 3 (2013-2015)

In [ ]:
primeiro_periodo = 2013.1
ultimo_periodo = 2015.2

marco_titulo <- paste("(", primeiro_periodo, " - ", ultimo_periodo, ")")

aux <- filtra_alunos_por_ingresso(alunos, primeiro_periodo, ultimo_periodo)

total_periodos = min(aux$maior_periodo_evasao)

print(sprintf("%s: %i", "Total de períodos analisados", total_periodos))

alunos_marco_3_raw <- alunos %>%
    mutate_at(vars(evadiu_periodo), ~replace_na(., 0)) %>%
    filter(periodo_ingresso_concat >= primeiro_periodo) %>%
    filter(periodo_ingresso_concat <= ultimo_periodo) %>%
    filter(evadiu_periodo <= total_periodos)

alunos_marco_3_taxas <- calcula_taxas_base(alunos_marco_3_raw)
alunos_marco_3_taxas["proporcao_mulheres"] = alunos_marco_3_taxas$proporcao_mulheres[1]
alunos_marco_3_taxas <- calcula_taxas_inep(alunos_marco_3_taxas)

dado_raw <- alunos_marco_3_raw
dado_taxas <- alunos_marco_3_taxas

dado_bp <- dado_raw %>%
    filter(regular == FALSE) %>%
    mutate(situacao = ifelse(graduado == TRUE, "graduado", "desistente"))%>%
    mutate(situacao_sexo = paste(str_to_title(situacao), sexo))

boxplots <- gera_boxplot(dado_bp, marco_titulo)
print(boxplots[[1]])
print(boxplots[[2]])

resultado_novo <- calcula_bootstrap_ci(dado_raw, marco_titulo) %>% mutate(marco = marco_titulo)
resultado <- rbind(resultado, resultado_novo)
resultado

diff_semestre_evasao_df_novo <- diff_semestre_evasao_bootstrap(dado_raw) %>% mutate(marco = marco_titulo)
diff_semestre_evasao_df <- rbind(diff_semestre_evasao_df, diff_semestre_evasao_df_novo)
diff_semestre_evasao_df

In [ ]:
homens <- dado_raw %>%
  filter(sexo == "Homem")

mulheres <- dado_raw %>%
  filter(sexo == "Mulher")

ci_homens <- proporcao_cl_boot(homens$status) %>%
  mutate(sexo = "Homens")

ci_mulheres <- proporcao_cl_boot(mulheres$status) %>%
  mutate(sexo = "Mulheres")

evasao_ic_m3 <- rbind(ci_homens, ci_mulheres)

grafico_evasao_ic(evasao_ic_m3, "(2013-2015)")

----------------------

# Marco 4 (2016-2017)

In [ ]:
primeiro_periodo = 2016.1
ultimo_periodo = 2017.2

marco_titulo <- paste("(", primeiro_periodo, " - ", ultimo_periodo, ")")

aux <- filtra_alunos_por_ingresso(alunos, primeiro_periodo, ultimo_periodo)

total_periodos = min(aux$maior_periodo_evasao)

print(sprintf("%s: %i", "Total de períodos analisados", total_periodos))

alunos_marco_4_raw <- alunos %>%
    mutate_at(vars(evadiu_periodo), ~replace_na(., 0)) %>%
    filter(periodo_ingresso_concat >= primeiro_periodo) %>%
    filter(periodo_ingresso_concat <= ultimo_periodo) %>%
    filter(evadiu_periodo <= total_periodos)

alunos_marco_4_taxas <- calcula_taxas_base(alunos_marco_4_raw)
alunos_marco_4_taxas["proporcao_mulheres"] = alunos_marco_4_taxas$proporcao_mulheres[1]
alunos_marco_4_taxas <- calcula_taxas_inep(alunos_marco_4_taxas)

dado_raw <- alunos_marco_4_raw
dado_taxas <- alunos_marco_4_taxas

dado_bp <- dado_raw %>%
    filter(regular == FALSE) %>%
    mutate(situacao = ifelse(graduado == TRUE, "graduado", "desistente"))%>%
    mutate(situacao_sexo = paste(str_to_title(situacao), sexo))

boxplots <- gera_boxplot(dado_bp, marco_titulo)
print(boxplots[[1]])
print(boxplots[[2]])

resultado_novo <- calcula_bootstrap_ci(dado_raw, marco_titulo) %>% mutate(marco = marco_titulo)
resultado <- rbind(resultado, resultado_novo)
resultado

diff_semestre_evasao_df_novo <- diff_semestre_evasao_bootstrap(dado_raw) %>% mutate(marco = marco_titulo)
diff_semestre_evasao_df <- rbind(diff_semestre_evasao_df, diff_semestre_evasao_df_novo)
diff_semestre_evasao_df

In [ ]:
homens <- dado_raw %>%
  filter(sexo == "Homem")

mulheres <- dado_raw %>%
  filter(sexo == "Mulher")

ci_homens <- proporcao_cl_boot(homens$status) %>%
  mutate(sexo = "Homens")

ci_mulheres <- proporcao_cl_boot(mulheres$status) %>%
  mutate(sexo = "Mulheres")

evasao_ic_m4 <- rbind(ci_homens, ci_mulheres)

grafico_evasao_ic(evasao_ic_m4, "(2016-2017)")

-------------------

# Outros

In [ ]:
gera_boxplot_longitudinal <- function(dado, titulo) {
    bp <- ggplot(dado, aes(x = periodo_ingresso_concat, y = evadiu_periodo)) +
        boxplot_configs +
        stat_summary(fun.data = median_cl_boot, geom = "errorbar",
        colour = stat_summary_color) + stat_summary(fun.y = median, geom = "point", colour = stat_summary_color) +
        stat_summary(aes(label = round(after_stat(y), 1)), fun.y = median, geom = "text", size = 0, vjust = -0.5, hjust = 2) +
        # stat_summary(aes(label = round(after_stat(y), 1)), fun.y = function(x) quantile(x, 0.75), geom = "text", size = 2, vjust = 0, hjust= 0) +
        # stat_summary(aes(label = round(after_stat(y), 1)), fun.y = function(x) quantile(x, 0.25), geom = "text", size = 2, vjust = 0, hjust = 0) +
        ylab("Semestre de evasão") +
        xlab("Turma de ingresso") +
        ggtitle(titulo) +
        scale_fill + scale_color +
        tema +
        theme(
          plot.title=element_text(family='', face='bold', colour='black', size=15),
          panel.grid.major = element_line(colour = "grey"),
          axis.text.x = element_text(angle = 90, vjust = 0.5, hjust=1),
          axis.title = element_text(size = rel(1.2))
        )

    return(bp)
}

In [ ]:
primeiro_periodo = 2006.1
ultimo_periodo = 2014.2

dado_raw <- alunos %>%
    mutate_at(vars(evadiu_periodo), ~replace_na(., 0)) %>%
    filter(periodo_ingresso_concat >= primeiro_periodo) %>%
    filter(periodo_ingresso_concat <= ultimo_periodo) %>%
    mutate(periodo_ingresso_concat = sprintf("%0.1f", periodo_ingresso_concat))

print(gera_boxplot_longitudinal(dado_raw %>% filter(graduado == TRUE), "Semestre de conclusão por turmas"))
print(gera_boxplot_longitudinal(dado_raw %>% filter(desistente == TRUE), "Semestre de desistência por turmas"))

In [ ]:
# Create data:
data <- data.frame(
  year=rep(seq(1990,2016), each=1),
  minn=sample( seq(0,1,0.0001) , 27),
  maxx=sample( seq(1,2,0.0001) , 27)
)

glimpse(dado_raw)

dado_h <- dado_raw %>% filter(sexo == "Homem")
dado_m <- dado_raw %>% filter(sexo == "Mulher")
h <- calcula_bootstrap_desistente_graduado(dado_h) %>%
        rename(status = situacao) %>%
        mutate(status = tolower(status)) %>%
        mutate(low = estatistica-low) %>%
        mutate(high = high-estatistica) %>%
        reframe(sexo, status, low, high)
m <- calcula_bootstrap_desistente_graduado(dado_m) %>%
        rename(status = situacao) %>%
        mutate(status = tolower(status)) %>%
        mutate(low = estatistica-low) %>%
        mutate(high = high-estatistica) %>%
        reframe(sexo, status, low, high)

ci_sexos <- rbind(h, m)

resultados <- data.frame()
for (periodo in c((dado_raw %>% filter(!duplicated(periodo_ingresso_concat)))$periodo_ingresso_concat)) {
    medias <- dado_raw %>%
        filter(regular == FALSE) %>%
        filter(periodo_ingresso_concat == periodo) %>%
        group_by(sexo, status) %>%
        mutate(media = mean(evadiu_periodo)) %>%
        filter(!duplicated(media))

        resultados <- rbind(resultados, medias)
}

resultados <- resultados %>%
    reframe(sexo, periodo_ingresso_concat, media, status) %>%
    rename(periodo_ingresso = periodo_ingresso_concat)

dado_ci <- full_join(ci_sexos,resultados,by=c("sexo", "status"))

conclusao_longitudinal_sexo_marco <- resultado %>%
    filter(situacao == "Graduado") %>%
    filter(sexo != "Todos") %>%
    ggplot(aes(x=marco, group = sexo)) +
    geom_ribbon(aes(fill = sexo, ymin=low,ymax=high), alpha = 0.2) +
    geom_line(aes(y=estatistica, color = sexo), alpha=0.8) +
    ylab("Semestre de evasão") +
    xlab("Turma de ingresso") +
    ggtitle("Semestre de evasão de Concluintes por sexo e marco") +
    scale_fill + scale_color +
    tema +
    theme(
      plot.title=element_text(family='', face='bold', colour='black', size=15),
      panel.grid.major = element_line(colour = "grey"),
      axis.text.x = element_text(angle = 90, vjust = 0.5, hjust=1),
      axis.title = element_text(size = rel(1.2))
    )

print(conclusao_longitudinal_sexo_marco)

evasao_longitudinal_sexo_marco <- resultado %>%
    filter(situacao == "Desistente") %>%
    filter(sexo != "Todos") %>%
    ggplot(aes(x=marco, group = sexo)) +
    geom_ribbon(aes(ymin=low,ymax=high, fill = sexo), alpha = 0.2) +
    geom_line(aes(y=estatistica, color = sexo), alpha=0.8) +
    ylab("Semestre de evasão") +
    xlab("Turma de ingresso") +
    ggtitle("Semestre de evasão de Desistentes por sexo e marco") +
    scale_fill + scale_color +
    tema +
    theme(
      plot.title=element_text(family='', face='bold', colour='black', size=15),
      panel.grid.major = element_line(colour = "grey"),
      axis.text.x = element_text(angle = 90, vjust = 0.5, hjust=1),
      axis.title = element_text(size = rel(1.2))
    )

print(evasao_longitudinal_sexo_marco)

-------------------

# Visualizações

## Função gera visualizações TAXAS INEP

indicadores_fluxo_basico, desistencia_semestre, concluintes_semestre, dados_de_sucesso, dados_metricas_sexo

In [ ]:
gera_visualizacoes_taxas_inep <- function (dados_raw, dados_taxas, sexo_filtro, primeiro_periodo, ultimo_periodo) {
    primeiro_periodo = sprintf("%1.1f", primeiro_periodo, 2)
    ultimo_periodo = sprintf("%1.1f", ultimo_periodo, 2)
    sexo_titulo = ifelse(sexo_filtro == "Mulher", "Mulheres", ifelse(sexo_filtro == "Homem", "Homens", "Todos"))

    indicadores_fluxo_basico <- dados_taxas %>%
        arrange(evadiu_periodo) %>%
        filter(sexo == sexo_filtro) %>%
        ggplot(aes(x = factor(evadiu_periodo), group = 1)) +
        geom_line(aes(y = taxa_permanencia, color = "taxa permanencia"), size = 1.2) +
        geom_line(aes(y = taxa_desistencia_acumulada, color = "taxa desistencia acumulada"), size = 1.2) +
        geom_line(aes(y = taxa_conclusao_acumulada, color = "taxa conclusao acumulada"), size = 1.2) +
    
        labs(color = 'Indicadores de fluxo') +
        ggtitle(paste(sexo_titulo)) +
        xlab("Semestre") +
        ylab("Taxas") +
        coord_cartesian(ylim = c(0, 100)) +
        scale_fill + scale_color +
        tema +
        theme(
          plot.title=element_text(family='', face='bold', colour='black', size=15),
          panel.grid.major = element_line(colour = "grey"),
          axis.title = element_text(size = rel(1.2))
        )

    maior_desistencia <- dados_taxas %>%
        mutate(maior = max(taxa_desistencia_periodo)) %>%
        reframe(maior) %>%
        filter(!duplicated(maior))

    desistencia_semestre <- dados_taxas %>%
        arrange(evadiu_periodo) %>%
        filter(sexo == sexo_filtro) %>%
        ggplot(aes(x = factor(evadiu_periodo), y = taxa_desistencia_periodo)) +
        geom_bar(stat="identity") +
        ggtitle(paste(sexo_titulo)) +
        xlab("Semestre") +
        ylab("Taxa de desistência") +
        coord_cartesian(ylim = c(0, maior_desistencia$maior[1])) +
        scale_fill + scale_color +
        tema +
        theme(
          plot.title=element_text(family='', face='bold', colour='black', size=15),
          panel.grid.major = element_line(colour = "grey"),
          axis.title = element_text(size = rel(1.2))
        )

    maior_conclusao <- dados_taxas %>%
        mutate(maior = max(taxa_conclusao_periodo)) %>%
        reframe(maior) %>%
        filter(!duplicated(maior))

    concluintes_semestre <- dados_taxas %>%
        arrange(evadiu_periodo) %>%
        filter(sexo == sexo_filtro) %>%
        ggplot(aes(x = factor(evadiu_periodo), y = taxa_conclusao_periodo)) +
        geom_bar(stat="identity") +
        ggtitle(paste(sexo_titulo)) +
        xlab("Semestre") +
        ylab("Taxa de conclusão") +
        coord_cartesian(ylim = c(0, maior_conclusao$maior[1])) +
        scale_fill + scale_color +
        tema +
        theme(
          plot.title=element_text(family='', face='bold', colour='black', size=15),
          panel.grid.major = element_line(colour = "grey"),
          axis.title = element_text(size = rel(1.2))
        )

    dados_de_sucesso <- dados_taxas %>%
        reframe(sexo, taxa_eficiencia, taxa_sucesso, taxa_sucesso_final) %>%
        filter(!duplicated(taxa_eficiencia))

    dados_raw_todos <- dados_raw %>%
        mutate(sexo = "Todos")

    dados_raw <- rbind(dados_raw, dados_raw_todos)
    proporcao_mulheres <- dados_taxas$proporcao_mulheres[1]
    dados_raw["proporcao_mulheres"] = proporcao_mulheres

    dados_metricas_sexo <- dados_raw %>%
        group_by(sexo) %>%
        filter(graduado == TRUE) %>%
        mutate(total_ingressos = n()) %>%
        mutate(conclusao_total = sum(evadiu_periodo)) %>%
        mutate(media_semestres_conclusao = conclusao_total/total_ingressos) %>%
        mutate(mediana_semestres_conclusao = median(evadiu_periodo)) %>%
        mutate(moda_semestres_conclusao = getmode(evadiu_periodo)) %>%
        mutate(desvio_padrao_semestres_conclusao = sd(evadiu_periodo)) %>%
        reframe(sexo, media_semestres_conclusao, mediana_semestres_conclusao, moda_semestres_conclusao, desvio_padrao_semestres_conclusao, total_ingressos, proporcao_mulheres) %>%
        filter(!duplicated(media_semestres_conclusao))

    return(list(indicadores_fluxo_basico, desistencia_semestre, concluintes_semestre, dados_de_sucesso, dados_metricas_sexo))
}

## Função gera visualizações CRA
taxas_cras, taxas_cras_graduados, taxas_cras_desistentes, visualizacao_cras, visualizacao_cras_desistentes, visualizacao_cras_graduados

In [ ]:
gera_visualizacoes_cra <- function(sexo_filtro, dados_raw) {
    if (sexo_filtro == "Todos") {
        dados_raw <- dados_raw %>%
            mutate(sexo = "Todos")
    }

    dados_raw_sexo <- dados_raw %>% filter(sexo == sexo_filtro)

    quartis_gerais <- quantile(dados_raw_sexo$cra, probs=c(0, 0.25, 0.75, 1))

    sexo_titulo = ifelse(sexo_filtro == "Mulher", "(Mulheres)", ifelse(sexo_filtro == "Homem", "(Homens)", "(Todos)"))

    dados_raw_sexo <- dados_raw_sexo %>%
        within(quartil <- as.integer(.bincode(cra, quantile(cra, probs=c(0, 0.25, 0.75, 1)), include.lowest=TRUE))) %>%
        mutate(quartil = ifelse(quartil == 1, "0-25", ifelse(quartil == 2, "26-75", "76-100")))

    taxas_cras <- dados_raw_sexo %>%
        mutate(status = "Todos") %>%
        mutate(total_ingressos = n()) %>%
        mutate(total_cras = sum(cra)) %>%
        mutate(media = total_cras/total_ingressos) %>%
        mutate(mediana = median(cra)) %>%
        mutate(moda = getmode(cra)) %>%
        mutate(desvio_padrao = sd(cra)) %>%
        mutate(minimo = min(cra)) %>%
        mutate(maximo = max(cra)) %>%
        mutate(quartil_2 = quartis_gerais[2]) %>%
        mutate(quartil_3 = quartis_gerais[3]) %>%
        reframe(sexo, status, media, mediana, moda, desvio_padrao, minimo, maximo, quartil_2, quartil_3) %>%
        filter(!duplicated(media))

    taxas_cras_graduados <- dados_raw_sexo %>%
        filter(graduado == TRUE) %>%
        mutate(status = "Graduado") %>%
        mutate(total_ingressos = n()) %>%
        mutate(total_cras = sum(cra)) %>%
        mutate(media = total_cras/total_ingressos) %>%
        mutate(mediana = median(cra)) %>%
        mutate(moda = getmode(cra)) %>%
        mutate(desvio_padrao = sd(cra)) %>%
        mutate(minimo = min(cra)) %>%
        mutate(maximo = max(cra)) %>%
        reframe(sexo, status, media, mediana, moda, desvio_padrao, minimo, maximo) %>%
        filter(!duplicated(media))

    taxas_cras_desistentes <- dados_raw_sexo %>%
        filter(desistente == TRUE) %>%
        mutate(status = "Desistente") %>%
        mutate(total_ingressos = n()) %>%
        mutate(total_cras = sum(cra)) %>%
        mutate(media = total_cras/total_ingressos) %>%
        mutate(mediana = median(cra)) %>%
        mutate(moda = getmode(cra)) %>%
        mutate(desvio_padrao = sd(cra)) %>%
        mutate(minimo = min(cra)) %>%
        mutate(maximo = max(cra)) %>%
        reframe(sexo, status, media, mediana, moda, desvio_padrao, minimo, maximo) %>%
        filter(!duplicated(media))

    visualizacao_cras <- dados_raw_sexo %>%
        mutate(total_ingressos = n()) %>%
        mutate(cras = sum(cra)) %>%
        mutate(media = cras/total_ingressos) %>%
        mutate(mediana = median(cra)) %>%
        within(quartil <- as.integer(.bincode(cra, quantile(cra, probs=c(0, 0.25, 0.75, 1)), include.lowest=TRUE))) %>%
        mutate(quartil = ifelse(quartil == 1, "0-25", ifelse(quartil == 2, "26-75", "76-100"))) %>%
        ggplot(aes(x = cra)) +
        geom_histogram(aes(y=..count../sum(..count..) * 100), bins = 20) +
        geom_vline(xintercept = taxas_cras$media[1], color = stat_summary_color) +
        ggtitle(paste(sexo_titulo)) +
        xlab("CRA") +
        ylab("Porcentagem") +
        coord_cartesian(xlim = c(0, 10)) +
        scale_fill + scale_color +
        tema +
        theme(
          plot.title=element_text(family='', face='bold', colour='black', size=15),
          panel.grid.major = element_line(colour = "grey"),
          axis.title = element_text(size = rel(1.2))
        )

    visualizacao_cras_desistentes <- dados_raw_sexo %>%
        filter(desistente == TRUE) %>%
        mutate(total_ingressos = n()) %>%
        mutate(cras = sum(cra)) %>%
        mutate(media = cras/total_ingressos) %>%
        mutate(mediana = median(cra)) %>%
        within(quartil <- as.integer(.bincode(cra, quantile(cra, probs=c(0, 0.25, 0.75, 1)), include.lowest=TRUE))) %>%
        mutate(quartil = ifelse(quartil == 1, "0-25", ifelse(quartil == 2, "26-75", "76-100"))) %>%
        ggplot(aes(x = cra)) +
        geom_histogram(aes(y=..count../sum(..count..) * 100), bins = 20) +
        geom_vline(xintercept = taxas_cras_desistentes$media[1], color = stat_summary_color) +
        ggtitle(paste(sexo_titulo)) +
        xlab("CRA") +
        ylab("Porcentagem") +
        coord_cartesian(xlim = c(0, 10)) +
        scale_fill + scale_color +
        tema +
        theme(
          plot.title=element_text(family='', face='bold', colour='black', size=15),
          panel.grid.major = element_line(colour = "grey"),
          axis.title = element_text(size = rel(1.2)))

    visualizacao_cras_graduados <- dados_raw_sexo %>%
        filter(graduado == TRUE) %>%
        mutate(total_ingressos = n()) %>%
        mutate(cras = sum(cra)) %>%
        mutate(media = cras/total_ingressos) %>%
        mutate(mediana = median(cra)) %>%
        within(quartil <- as.integer(.bincode(cra, quantile(cra, probs=c(0, 0.25, 0.75, 1)), include.lowest=TRUE))) %>%
        mutate(quartil = ifelse(quartil == 1, "0-25", ifelse(quartil == 2, "26-75", "76-100"))) %>%
        ggplot(aes(x = cra)) +
        geom_histogram(aes(y=..count../sum(..count..) * 100), bins = 20) +
        geom_vline(xintercept = taxas_cras_graduados$media[1], color = stat_summary_color) +
        ggtitle(paste(sexo_titulo)) +
        xlab("CRA") +
        ylab("Porcentagem") +
        coord_cartesian(xlim = c(0, 10)) +
        scale_fill + scale_color +
        tema +
        theme(
          plot.title=element_text(family='', face='bold', colour='black', size=15),
          panel.grid.major = element_line(colour = "grey"),
          axis.title = element_text(size = rel(1.2))
        )

    return(list(taxas_cras, taxas_cras_graduados, taxas_cras_desistentes, visualizacao_cras, visualizacao_cras_desistentes, visualizacao_cras_graduados))
}

------------------------------------------

# CRA

In [ ]:
cra_agregado <- alunos_agregados_raw %>% mutate(marco = "Agregado") %>% reframe(sexo, cra, marco, graduado, regular, desistente)
cra_marco_1 <- alunos_marco_1_raw %>% mutate(marco = "Marco 1") %>% reframe(sexo, cra, marco, graduado, regular, desistente)
cra_marco_2 <- alunos_marco_2_raw %>% mutate(marco = "Marco 2") %>% reframe(sexo, cra, marco, graduado, regular, desistente)
cra_marco_3 <- alunos_marco_3_raw %>% mutate(marco = "Marco 3") %>% reframe(sexo, cra, marco, graduado, regular, desistente)
cra_marco_4 <- alunos_marco_4_raw %>% mutate(marco = "Marco 4") %>% reframe(sexo, cra, marco, graduado, regular, desistente)

alunos_raw_cra <- rbind(cra_agregado, cra_marco_1)
alunos_raw_cra <- rbind(alunos_raw_cra, cra_marco_2)
alunos_raw_cra <- rbind(alunos_raw_cra, cra_marco_3)
alunos_raw_cra <- rbind(alunos_raw_cra, cra_marco_4)
alunos_raw_cra_todos <- alunos_raw_cra %>% mutate(sexo = "Todos")
alunos_raw_cra <- rbind(alunos_raw_cra, alunos_raw_cra_todos)

alunos_raw_cra <- alunos_raw_cra %>%
   gather(key="status", value="estado", 4:6) %>%
   filter(estado == TRUE) %>%
   select(-estado)

glimpse(alunos_raw_cra)

## Agregado (2006.1 - 2014.2)

In [ ]:
primeiro_periodo = 2006.1
ultimo_periodo = 2014.2
dados_raw <- alunos_agregados_raw
dados_taxas <- alunos_agregados_taxas

In [ ]:
### Todos

visualizacoes_inep_todos <- gera_visualizacoes_taxas_inep(dados_raw, dados_taxas, "Todos", primeiro_periodo, ultimo_periodo)

dados_raw %>%
    filter(!duplicated(sexo))

visualizacoes_cra_todos <- gera_visualizacoes_cra("Todos", dados_raw)

### Mulheres

visualizacoes_inep_mulher <- gera_visualizacoes_taxas_inep(dados_raw, dados_taxas, "Mulher", primeiro_periodo, ultimo_periodo)

visualizacoes_cra_mulher <- gera_visualizacoes_cra("Mulher", dados_raw)

### Homens

visualizacoes_inep_homem <- gera_visualizacoes_taxas_inep(dados_raw, dados_taxas, "Homem", primeiro_periodo, ultimo_periodo)

visualizacoes_cra_homem <- gera_visualizacoes_cra("Homem", dados_raw)

## Marco 1 (2006.1 - 2008.2)

In [ ]:
primeiro_periodo = 2006.1
ultimo_periodo = 2008.2
dados_raw <- alunos_marco_1_raw
dados_taxas <- alunos_marco_1_taxas

In [ ]:
### Todos

visualizacoes_inep_todos_m1 <- gera_visualizacoes_taxas_inep(dados_raw, dados_taxas, "Todos", primeiro_periodo, ultimo_periodo)

visualizacoes_cra_todos_m1 <- gera_visualizacoes_cra("Todos", dados_raw)

### Mulheres

visualizacoes_inep_mulher_m1 <- gera_visualizacoes_taxas_inep(dados_raw, dados_taxas, "Mulher", primeiro_periodo, ultimo_periodo)

visualizacoes_cra_mulher_m1 <- gera_visualizacoes_cra("Mulher", dados_raw)

### Homens

visualizacoes_inep_homem_m1 <- gera_visualizacoes_taxas_inep(dados_raw, dados_taxas, "Homem", primeiro_periodo, ultimo_periodo)

visualizacoes_cra_homem_m1 <- gera_visualizacoes_cra("Homem", dados_raw)

## Marco 2 (2009.1 - 2012.2)

In [ ]:
primeiro_periodo = 2009.1
ultimo_periodo = 2012.2
dados_raw <- alunos_marco_2_raw
dados_taxas <- alunos_marco_2_taxas

In [ ]:
### Todos

visualizacoes_inep_todos_m2 <- gera_visualizacoes_taxas_inep(dados_raw, dados_taxas, "Todos", primeiro_periodo, ultimo_periodo)

visualizacoes_cra_todos_m2 <- gera_visualizacoes_cra("Todos", dados_raw)

### Mulheres

visualizacoes_inep_mulher_m2 <- gera_visualizacoes_taxas_inep(dados_raw, dados_taxas, "Mulher", primeiro_periodo, ultimo_periodo)

visualizacoes_cra_mulher_m2 <- gera_visualizacoes_cra("Mulher", dados_raw)

### Homens

visualizacoes_inep_homem_m2 <- gera_visualizacoes_taxas_inep(dados_raw, dados_taxas, "Homem", primeiro_periodo, ultimo_periodo)

visualizacoes_cra_homem_m2 <- gera_visualizacoes_cra("Homem", dados_raw)

## Marco 3 (2013.1 - 2015.2)

In [ ]:
primeiro_periodo = 2013.1
ultimo_periodo = 2015.2
dados_raw <- alunos_marco_3_raw
dados_taxas <- alunos_marco_3_taxas

In [ ]:
### Todos

visualizacoes_inep_todos_m3 <- gera_visualizacoes_taxas_inep(dados_raw, dados_taxas, "Todos", primeiro_periodo, ultimo_periodo)

visualizacoes_cra_todos_m3 <- gera_visualizacoes_cra("Todos", alunos_agregados_raw)

### Mulheres

visualizacoes_inep_mulher_m3 <- gera_visualizacoes_taxas_inep(dados_raw, dados_taxas, "Mulher", primeiro_periodo, ultimo_periodo)

visualizacoes_cra_mulher_m3 <- gera_visualizacoes_cra("Mulher", alunos_agregados_raw)

### Homens

visualizacoes_inep_homem_m3 <- gera_visualizacoes_taxas_inep(dados_raw, dados_taxas, "Homem", primeiro_periodo, ultimo_periodo)

visualizacoes_cra_homem_m3 <- gera_visualizacoes_cra("Homem", alunos_agregados_raw)

## Marco 4 (2016.1 - 2017.2)

In [ ]:
primeiro_periodo = 2016.1
ultimo_periodo = 2017.2
dados_raw <- alunos_marco_4_raw
dados_taxas <- alunos_marco_4_taxas

In [ ]:
### Todos

visualizacoes_inep_todos_m4 <- gera_visualizacoes_taxas_inep(dados_raw, alunos_marco_4_taxas, "Todos", primeiro_periodo, ultimo_periodo)

visualizacoes_cra_todos_m4 <- gera_visualizacoes_cra("Todos", dados_raw)

### Mulheres

visualizacoes_inep_mulher_m4 <- gera_visualizacoes_taxas_inep(dados_raw, alunos_marco_4_taxas, "Mulher", primeiro_periodo, ultimo_periodo)

visualizacoes_cra_mulher_m4 <- gera_visualizacoes_cra("Mulher", dados_raw)

### Homens

visualizacoes_inep_homem_m4 <- gera_visualizacoes_taxas_inep(dados_raw, alunos_marco_4_taxas, "Homem", primeiro_periodo, ultimo_periodo)

visualizacoes_cra_homem_m4 <- gera_visualizacoes_cra("Homem", dados_raw)

## Bootstrap de todos os CRAs

In [ ]:
boxplot_cras_group_sexo <- function (dado, marco_filtro, titulo) {
    bp <- dado %>%
        filter(marco == marco_filtro) %>%
        ggplot(aes(x = sexo, y = cra, fill = sexo)) +
        geom_boxplot(
            # custom boxes
            color="ivory4",
            fill="ivory4",
            alpha=0.4,

            outlier.size=3
        ) +
        stat_summary(fun.data = median_cl_boot, geom = "errorbar",
        colour = stat_summary_color) + stat_summary(fun.y = median, geom = "point", colour = stat_summary_color) +
        stat_summary(aes(label = round(after_stat(y), 1)), fun.y = median, geom = "text", size = 4, vjust = -0.5, hjust = 2) +
        stat_summary(aes(label = round(after_stat(y), 1)), fun.y = function(x) quantile(x, 0.75), geom = "text", size = 4, vjust = -1, hjust= 2) +
        stat_summary(aes(label = round(after_stat(y), 1)), fun.y = function(x) quantile(x, 0.25), geom = "text", size = 4, vjust = 2, hjust = 2) +
        ylab("CRA") +
        xlab("Sexo") +
        ggtitle(titulo) +
        scale_fill + scale_color +
        tema +
        theme(
          plot.title=element_text(family='', face='bold', colour='black', size=15),
          panel.grid.major = element_line(colour = "grey"),
          axis.text.x = element_text(angle = 90, vjust = 0.5, hjust=1),
          axis.title = element_text(size = rel(1.2))
        )

    return(bp)

}

boxplot_cras_group_marco <- function (dado, sexo_filtro, titulo) {
    bp <- dado %>%
        filter(sexo == sexo_filtro) %>%
        ggplot(aes(x = marco, y = cra)) +
        boxplot_configs +
        stat_summary(fun.data = median_cl_boot, geom = "errorbar",
        colour = stat_summary_color) + stat_summary(fun.y = median, geom = "point", colour = stat_summary_color) +
        stat_summary(aes(label = round(after_stat(y), 1)), fun.y = median, geom = "text", size = 0, vjust = -0.5, hjust = 2) +
        stat_summary(aes(label = round(after_stat(y), 1)), fun.y = function(x) quantile(x, 0.75), geom = "text", size = 2, vjust = -1, hjust= 2) +
        stat_summary(aes(label = round(after_stat(y), 1)), fun.y = function(x) quantile(x, 0.25), geom = "text", size = 2, vjust = 2, hjust = 2) +
        ylab("CRA") +
        xlab("Marco") +
        ggtitle(paste("CRAs", titulo)) +
        scale_fill + scale_color +
        tema +
        theme(
          plot.title=element_text(family='', face='bold', colour='black', size=15),
          panel.grid.major = element_line(colour = "grey"),
          axis.title = element_text(size = rel(1.2))
        )
    return(bp)
}

boxplot_cras_group_status <- function (dado, marco_filtro, titulo) {
    bp <- dado %>%
        filter(marco == marco_filtro) %>%
        ggplot(aes(x = status, y = cra)) +
        boxplot_configs +
        stat_summary(fun.data = median_cl_boot, geom = "errorbar",
        colour = stat_summary_color) + stat_summary(fun.y = median, geom = "point", colour = stat_summary_color) +
        stat_summary(aes(label = round(after_stat(y), 1)), fun.y = median, geom = "text", size = 4, vjust = -0.5, hjust = 2) +
        stat_summary(aes(label = round(after_stat(y), 1)), fun.y = function(x) quantile(x, 0.75), geom = "text", size = 4, vjust = -1, hjust= 2) +
        stat_summary(aes(label = round(after_stat(y), 1)), fun.y = function(x) quantile(x, 0.25), geom = "text", size = 4, vjust = 2, hjust = 2) +
        ylab("CRA") +
        xlab("Status") +
        ggtitle(titulo) +
        scale_fill + scale_color +
        tema +
        theme(
          plot.title=element_text(family='', face='bold', colour='black', size=15),
          panel.grid.major = element_line(colour = "grey"),
          axis.title = element_text(size = rel(1.2))
        )
    return(bp)
}

boxplot_cras_group_marco_sexo <- function (dado, titulo) {
    bp <- dado %>%
        ggplot(aes(x = marco, y = cra, fill = sexo)) +
        geom_boxplot(aes(fill = sexo), alpha=0.3, outlier.size=3) +
        stat_summary(aes(group = interaction(marco, sexo)), fun.data = median_cl_boot, geom = "errorbar", position = position_dodge(0.8), color = stat_summary_color, alpha = 0.7) +
        ylab("CRA") +
        xlab("Marco") +
        ggtitle(paste("CRAs", titulo)) +
        scale_fill + scale_color +
        tema +
        theme(
          plot.title=element_text(family='', face='bold', colour='black', size=15),
          panel.grid.major = element_line(colour = "grey"),
          axis.title = element_text(size = rel(1.2)),
          legend.title = element_text(angle = 90, hjust = 0.5),
          legend.position = "bottom"
        )
    return(bp)
}

## Boxplot dos CRAs

In [ ]:
marcos <- list("Agregado", "Marco 1", "Marco 2", "Marco 3", "Marco 4")

for (marco in marcos) {
    um <- boxplot_cras_group_status(alunos_raw_cra, marco, "Por status")
    dois <- boxplot_cras_group_sexo(alunos_raw_cra, marco, "Todos os ingressantes")
    tres <- boxplot_cras_group_sexo(alunos_raw_cra %>% filter(status == "desistente"), marco, "Desistentes")
    quatro <- boxplot_cras_group_sexo(alunos_raw_cra %>% filter(status == "graduado"), marco, "Concluintes")

    combined <- um + dois + tres + quatro +
      plot_annotation(tag_levels = "A", title = paste("Distribuição de CRAs -", str_to_title(marco))) +
      plot_layout(guides = "collect", ncol = 2) +
      tema +
      theme(
        plot.title=element_text(family='', face='bold', colour='black', size=15),
        panel.grid.major = element_line(colour = "grey"),
        axis.title = element_text(size = rel(1.2))
      )

    print(combined)
}

In [ ]:
glimpse(alunos_raw_cra)

In [ ]:
print(boxplot_cras_group_marco(alunos_raw_cra, "Todos", "todos"))
print(boxplot_cras_group_marco(alunos_raw_cra, "Homem", "homens"))
print(boxplot_cras_group_marco(alunos_raw_cra, "Mulher", "mulheres"))

print(boxplot_cras_group_marco(alunos_raw_cra %>% filter(status == "graduado"), "Todos", "concluintes (todos)"))
print(boxplot_cras_group_marco(alunos_raw_cra %>% filter(status == "graduado"), "Homem", "homens concluintes"))
print(boxplot_cras_group_marco(alunos_raw_cra %>% filter(status == "graduado"), "Mulher", "mulheres concluintes"))

print(boxplot_cras_group_marco(alunos_raw_cra %>% filter(status == "desistente"), "Todos", "desistentes (todos)"))
print(boxplot_cras_group_marco(alunos_raw_cra %>% filter(status == "desistente"), "Homem", "homens desistentes"))
print(boxplot_cras_group_marco(alunos_raw_cra %>% filter(status == "desistente"), "Mulher", "mulheres desistentes"))

print(boxplot_cras_group_marco_sexo(alunos_raw_cra %>% filter(status == "desistente") %>% filter(sexo != "Todos"), "desistentes"))
print(boxplot_cras_group_marco_sexo(alunos_raw_cra %>% filter(status == "graduado") %>% filter(sexo != "Todos"), "concluintes"))

In [ ]:
plota_bp_cra_marco_e_estado <- function (dado, marco_filtro) {
  dado %>%
    filter(marco == marco_filtro) %>%
    ggplot(aes(x = status, y = cra)) +
    boxplot_configs +
    stat_summary(fun.data = median_cl_boot, geom = "errorbar",
    colour = stat_summary_color) + stat_summary(fun.y = median, geom = "point", colour = stat_summary_color) +
    stat_summary(aes(label = round(after_stat(y), 1)), fun.y = median, geom = "text", size = 4, vjust = -0.5, hjust = 2) +
    stat_summary(aes(label = round(after_stat(y), 1)), fun.y = function(x) quantile(x, 0.75), geom = "text", size = 4, vjust = -1, hjust= 2) +
    stat_summary(aes(label = round(after_stat(y), 1)), fun.y = function(x) quantile(x, 0.25), geom = "text", size = 4, vjust = 2, hjust = 2) +
    ylab("CRA") +
    xlab("Estado") +
    ggtitle(paste("CRAs -", marco_filtro)) +
    scale_fill + scale_color +
    tema +
    theme(
      plot.title=element_text(family='', face='bold', colour='black', size=15),
      panel.grid.major = element_line(colour = "grey"),
      axis.title = element_text(size = rel(1.2))
    )
}

print(plota_bp_cra_marco_e_estado(alunos_raw_cra %>% filter(sexo == "Todos"), "Agregado"))
print(plota_bp_cra_marco_e_estado(alunos_raw_cra %>% filter(status != "regular"), "Marco 1"))
print(plota_bp_cra_marco_e_estado(alunos_raw_cra, "Marco 2"))
print(plota_bp_cra_marco_e_estado(alunos_raw_cra, "Marco 3"))
print(plota_bp_cra_marco_e_estado(alunos_raw_cra, "Marco 4"))

-------------------

# Taxas INEP

### Agregado (2006.1 - 2014.2)

In [ ]:
p1 <- visualizacoes_inep_homem
p2 <- visualizacoes_inep_mulher
p3 <- visualizacoes_inep_todos

print(plota_junto(p1, p2, p3, 1, "Indicadores de fluxo básico (Turmas de 2006.1 a 2014.2)"))
print(plota_junto(p1, p2, p3, 2, "Taxa de desistência por semestre (Turmas de 2006.1 a 2014.2)"))
print(plota_junto(p1, p2, p3, 3, "Taxa de conclusão por semestre (Turmas de 2006.1 a 2014.2)"))

cat("\n\n\n Taxas de sucesso")
print(p3[[4]])

cat("\n\n\n Tempo de conclusão")
print(p3[[5]])

### Marco 1 (2006.1 - 2008.2)

In [ ]:
p1 <- visualizacoes_inep_homem_m1
p2 <- visualizacoes_inep_mulher_m1
p3 <- visualizacoes_inep_todos_m1

print(plota_junto(p1, p2, p3, 1, "Indicadores de fluxo básico (Turmas de 2006.1 - 2008.2)"))
print(plota_junto(p1, p2, p3, 2, "Taxa de desistência por semestre (Turmas de 2006.1 - 2008.2)"))
print(plota_junto(p1, p2, p3, 3, "Taxa de conclusão por semestre (Turmas de 2006.1 - 2008.2)"))

cat("\n\n\n Taxas de sucesso")
print(p3[[4]])

cat("\n\n\n Tempo de conclusão")
print(p3[[5]])

### Marco 2 (2009.1 - 2012.2)

In [ ]:
p1 <- visualizacoes_inep_homem_m2
p2 <- visualizacoes_inep_mulher_m2
p3 <- visualizacoes_inep_todos_m2

print(plota_junto(p1, p2, p3, 1, "Indicadores de fluxo básico (Turmas de 2009.1 - 2012.2)"))
print(plota_junto(p1, p2, p3, 2, "Taxa de desistência por semestre (Turmas de 2009.1 - 2012.2)"))
print(plota_junto(p1, p2, p3, 3, "Taxa de conclusão por semestre (Turmas de 2009.1 - 2012.2)"))

cat("\n\n\n Taxas de sucesso")
print(p3[[4]])

cat("\n\n\n Tempo de conclusão")
print(p3[[5]])

### Marco 3 (2013.1 - 2015.2)

In [ ]:
p1 <- visualizacoes_inep_homem_m3
p2 <- visualizacoes_inep_mulher_m3
p3 <- visualizacoes_inep_todos_m3

print(plota_junto(p1, p2, p3, 1, "Indicadores de fluxo básico (Turmas de 2013.1 - 2015.2)"))
print(plota_junto(p1, p2, p3, 2, "Taxa de desistência por semestre (Turmas de 2013.1 - 2015.2)"))
print(plota_junto(p1, p2, p3, 3, "Taxa de conclusão por semestre (Turmas de 2013.1 - 2015.2)"))

cat("\n\n\n Taxas de sucesso")
print(p3[[4]])

cat("\n\n\n Tempo de conclusão")
print(p3[[5]])

### Marco 4 (2016.1 - 2017.2)

In [ ]:
p1 <- visualizacoes_inep_homem_m4
p2 <- visualizacoes_inep_mulher_m4
p3 <- visualizacoes_inep_todos_m4

print(plota_junto(p1, p2, p3, 1, "Indicadores de fluxo básico (Turmas de 2016.1 - 2017.2)"))
print(plota_junto(p1, p2, p3, 2, "Taxa de desistência por semestre (Turmas de 2016.1 - 2017.2)"))
print(plota_junto(p1, p2, p3, 3, "Taxa de conclusão por semestre (Turmas de 2016.1 - 2017.2)"))

cat("\n\n\n Taxas de sucesso")
print(p3[[4]])

cat("\n\n\n Tempo de conclusão")
print(p3[[5]])

--------------------

# CRA

In [ ]:
tabela_junto <- function (p1, p2, p3, indice) {
    rbind(
        p1[[indice]],
        p2[[indice]],
        p3[[indice]]
    )
}

plota_junto_cra <- function (p1, p2, p3, titulo) {
    combined <- p1 + p2 +
      plot_annotation(tag_levels = "A") +
      plot_layout(guides = "collect", ncol = 2) &
      tema +
      theme(
        plot.title=element_text(family='', face='bold', colour='black', size=15),
        plot.tag = element_text(family = "EB Garamond", size = 8),
        legend.position = "none",
        panel.grid.major = element_line(colour = "grey"),
        axis.title = element_text(size = rel(1.2))
    )

    (combined / p3) +
      plot_annotation(tag_levels = "A", title = titulo) +
      tema +
      theme(
        plot.tag = element_text(family = "EB Garamond", size = 6),
        legend.position = "bottom",
        legend.text.align = 1,
        text = element_text(margin = margin(0, 3, 0, 2)),
        legend.title = element_text(margin = margin(0, 6, 0, 0)),
        plot.title=element_text(family='', face='bold', colour='black', size=15),
        panel.grid.major = element_line(colour = "grey"),
        axis.title = element_text(size = rel(1.2))
    )
}

## Agregado (2006.1 - 2014.2)

In [ ]:
p1 <- visualizacoes_cra_homem
p2 <- visualizacoes_cra_mulher
p3 <- visualizacoes_cra_todos

cat("\n\n\n Métricas CRAs - Dados agregados (2006.1 - 2014.2)")
tabela_junto(p1, p2, p3, 1)

cat("\n\n\n Métricas CRAs - Dados agregados (2006.1 - 2014.2)")
tabela_junto(p1, p2, p3, 2)

cat("\n\n\n Métricas CRAs - Dados agregados (2006.1 - 2014.2)")
tabela_junto(p1, p2, p3, 3)

print(plota_junto_cra(p1[[4]], p2[[4]], p3[[4]], "Distribuição dos CRAS (2006.1 - 2014.2)"))
print(plota_junto_cra(p1[[5]], p2[[5]], p3[[5]], "Distribuição dos CRAS dos desistentes (2006.1 - 2014.2)"))
print(plota_junto_cra(p1[[6]], p2[[6]], p3[[6]], "Distribuição dos CRAS dos concluintes (2006.1 - 2014.2)"))

## Marco 1 (2006.1 - 2008.2)

In [ ]:
p1 <- visualizacoes_cra_homem_m1
p2 <- visualizacoes_cra_mulher_m1
p3 <- visualizacoes_cra_todos_m1

cat("\n\n\n Métricas CRAs - Marco 1 (2006.1 - 2008.2)")
tabela_junto(p1, p2, p3, 1)

cat("\n\n\n Métricas CRAs - Marco 1 (2006.1 - 2008.2)")
tabela_junto(p1, p2, p3, 2)

cat("\n\n\n Métricas CRAs - Marco 1 (2006.1 - 2008.2)")
tabela_junto(p1, p2, p3, 3)


print(plota_junto_cra(p1[[4]], p2[[4]], p3[[4]], "Distribuição dos CRAS (2006.1 - 2008.2)"))
print(plota_junto_cra(p1[[5]], p2[[5]], p3[[5]], "Distribuição dos CRAS dos desistentes (2006.1 - 2008.2)"))
print(plota_junto_cra(p1[[6]], p2[[6]], p3[[6]], "Distribuição dos CRAS dos concluintes (2006.1 - 2008.2)"))

## Marco 2 (2009.1 - 2012.2)

In [ ]:
p1 <- visualizacoes_cra_homem_m2
p2 <- visualizacoes_cra_mulher_m2
p3 <- visualizacoes_cra_todos_m2

cat("\n\n\n Métricas CRAs - Marco 2 (2009.1 - 2012.2)")
tabela_junto(p1, p2, p3, 1)

cat("\n\n\n Métricas CRAs - Marco 2 (2009.1 - 2012.2)")
tabela_junto(p1, p2, p3, 2)

cat("\n\n\n Métricas CRAs - Marco 2 (2009.1 - 2012.2)")
tabela_junto(p1, p2, p3, 3)


print(plota_junto_cra(p1[[4]], p2[[4]], p3[[4]], "Distribuição dos CRAS (2009.1 - 2012.2)"))
print(plota_junto_cra(p1[[5]], p2[[5]], p3[[5]], "Distribuição dos CRAS dos desistentes (2009.1 - 2012.2)"))
print(plota_junto_cra(p1[[6]], p2[[6]], p3[[6]], "Distribuição dos CRAS dos concluintes (2009.1 - 2012.2)"))

## Marco 3 (2013.1 - 2015.2)

In [ ]:
p1 <- visualizacoes_cra_homem_m3
p2 <- visualizacoes_cra_mulher_m3
p3 <- visualizacoes_cra_todos_m3

cat("\n\n\n Métricas CRAs - Marco 3 (2013.1 - 2015.2)")
tabela_junto(p1, p2, p3, 1)

cat("\n\n\n Métricas CRAs - Marco 3 (2013.1 - 2015.2)")
tabela_junto(p1, p2, p3, 2)

cat("\n\n\n Métricas CRAs - Marco 3 (2013.1 - 2015.2)")
tabela_junto(p1, p2, p3, 3)


print(plota_junto_cra(p1[[4]], p2[[4]], p3[[4]], "Distribuição dos CRAS (2013.1 - 2015.2)"))
print(plota_junto_cra(p1[[5]], p2[[5]], p3[[5]], "Distribuição dos CRAS dos desistentes (2013.1 - 2015.2)"))
print(plota_junto_cra(p1[[6]], p2[[6]], p3[[6]], "Distribuição dos CRAS dos concluintes (2013.1 - 2015.2)"))

## Marco 4 (2016.1 - 2017.2)

In [ ]:
dado_rawp1 <- visualizacoes_cra_homem_m4
p2 <- visualizacoes_cra_mulher_m4
p3 <- visualizacoes_cra_todos_m4

cat("\n\n\n Métricas CRAs - Marco 4 (2016.1 - 2017.2)")
tabela_junto(p1, p2, p3, 1)

cat("\n\n\n Métricas CRAs - Marco 4 (2016.1 - 2017.2)")
tabela_junto(p1, p2, p3, 2)

cat("\n\n\n Métricas CRAs - Marco 4 (2016.1 - 2017.2)")
tabela_junto(p1, p2, p3, 3)

print(plota_junto_cra(p1[[4]], p2[[4]], p3[[4]], "Distribuição dos CRAS (2016.1 - 2017.2)"))
print(plota_junto_cra(p1[[5]], p2[[5]], p3[[5]], "Distribuição dos CRAS dos desistentes (2016.1 - 2017.2)"))
print(plota_junto_cra(p1[[6]], p2[[6]], p3[[6]], "Distribuição dos CRAS dos concluintes (2016.1 - 2017.2)"))

______________________________________________

# Exportando gráficos

In [ ]:
tema_para_exportar = tema + theme(
        plot.title=element_text(family='', face='bold', colour='black', size=10),
        plot.tag = element_text(family = "EB Garamond", size = 10),
        legend.position = "bottom",
        legend.text = element_text(size=8),
        legend.title = element_text(size=10),
        panel.grid.major = element_line(colour = "grey"),
        axis.title = element_text(size = rel(1)),
        axis.text.x = element_text(size = 8, hjust = 1),
        axis.text.y = element_text(size = 8, hjust = 1)
    )

plota_junto_4 <- function (p1, p2, p3, p4, titulo) {
    combined1 <- p1 + p2 +
      plot_annotation(tag_levels = "A") +
      plot_layout(guides = "collect", ncol = 2) &
      tema_para_exportar

    combined2 <- p3 + p4 +
      plot_annotation(tag_levels = "A") +
      plot_layout(guides = "collect", ncol = 2) &
      tema_para_exportar

    (combined1 / combined2) +
      plot_annotation(
          tag_levels = "A",
          title = titulo,
          theme = theme(
              plot.title=element_text(family='', face='bold', colour='black', size=10)
          )
      )
}

t_m1 <- "(2006.1 - 2008.2)"
t_m2 <- "(2009.1 - 2012.2)"
t_m3 <- "(2013.1 - 2015.2)"
t_m4 <- "(2016.1 - 2017.2)"

In [ ]:
salvar_plot <- function (grafico, titulo_arquivo, altura, largura, titulo_plot = NULL, tema_modificador = NULL) {
    
    grafico <- grafico +
        {if(!is.null(titulo_plot))ggtitle(titulo_plot)} +
        labs() +
        tema_para_exportar +
        tema_modificador
    
    titulo_png <- paste(titulo_arquivo, ".png", sep="")
    
    ggsave(titulo_png, 
           plot = grafico, 
           height=altura, width=largura, units="mm")

    
    img <- magick::image_read(paste('/kaggle/working/', titulo_png, sep=""))
    plot(img)
}

## Indicadores de fluxo básico (Turmas de 2006.1 a 2014.2)

In [ ]:
titulo_arquivo <- "indicadores_fluxo_basico"
titulo_plot <- "Indicadores de fluxo básico (Turmas de 2006.1 a 2014.2)"
grafico <- visualizacoes_inep_todos[[1]]

salvar_plot(grafico, titulo_arquivo, altura=100, largura=200, titulo_plot=titulo_plot)

## Indicadores de fluxo básico (MARCOS)

In [ ]:
titulo_arquivo <- "indicadores_fluxo_basico_marcos"
titulo_plot <- "Indicadores de fluxo básico"

title_theme <- theme(plot.title=element_text(family='', face='bold', colour='black', size=10))
p1 <- visualizacoes_inep_todos_m1[[1]] + ggtitle("2006.1 - 2008.2") + title_theme
p2 <- visualizacoes_inep_todos_m2[[1]] + ggtitle("2009.1 - 2012.2") + title_theme
p3 <- visualizacoes_inep_todos_m3[[1]] + ggtitle("2013.1 - 2015.2") + title_theme
p4 <- visualizacoes_inep_todos_m4[[1]] + ggtitle("2016.1 - 2017.2") + title_theme

grafico <- ((p1 + p2) / (p3 + p4)) +
    plot_annotation(
        tag_levels = "A",
        title = titulo_plot,
        theme = tema + theme(legend.position = "bottom", plot.title=element_text(family='', face='bold', colour='black', size=16))
      ) +
      plot_layout(guides = "collect")

salvar_plot(grafico, titulo_arquivo, altura=180, largura=220, titulo_plot)

## Taxa de desistência por semestre (Turmas de 2006.1 a 2014.2)

In [ ]:
titulo_arquivo <- "taxa_desistencia_semestre"
titulo_plot <- "Taxa de desistência por semestre (Turmas de 2006.1 a 2014.2)"
grafico <- visualizacoes_inep_todos[[2]]

salvar_plot(grafico, titulo_arquivo, altura=90, largura=150, titulo_plot)

## Taxa de conclusão por semestre (Turmas de 2006.1 a 2014.2)

In [ ]:
titulo_arquivo <- "taxa_conclusao_semestre"
titulo_plot <- "Taxa de conclusão por semestre (Turmas de 2006.1 a 2014.2)"
grafico <- visualizacoes_inep_todos[[3]]

salvar_plot(grafico, titulo_arquivo, altura=90, largura=150, titulo_plot)

In [ ]:
titulo_arquivo <- "boxplot_evasao_agregado"
titulo_plot <- "Semestre de evasão por tipo (Turmas de 2006.1 a 2014.2)"
grafico <- boxplot_agregado[[2]] +
    scale_y_discrete(name ="Semestre de evasão", 
                    limits=c(2,4,6,8,10,12,14))

salvar_plot(grafico, titulo_arquivo, altura=130, largura=150, titulo_plot)

## Semestre de evasão por tipo (Turmas de 2006.1 a 2014.2)

## Distribuição dos CRAs (Turmas de 2006.1 a 2014.2)

In [ ]:
titulo_arquivo <- "dist_cras_agregados"
titulo_plot <- "Distribuição dos CRAs (Turmas de 2006.1 a 2014.2)"

p1 <- visualizacoes_cra_todos[[5]] +
    ggtitle("Desistentes")

p2 <- visualizacoes_cra_todos[[6]] +
    ggtitle("Concluintes")

grafico <- (p2 / p1) +
        plot_annotation(
        tag_levels = "A",
        title = titulo_plot,
        theme = tema + theme(legend.position = "bottom")
      ) +
      plot_layout(guides = "collect")

salvar_plot(grafico, titulo_arquivo, altura=150, largura=150, titulo_plot)

## CRAs (Turmas de 2006.1 a 2014.2)

In [ ]:
titulo_arquivo <- "boxplot_cras_agregados"
titulo_plot <- "CRAs (Turmas de 2006.1 a 2014.2)"
grafico <- plota_bp_cra_marco_e_estado(
        alunos_raw_cra %>%
        filter(sexo == "Todos"), "Agregado"
    )

salvar_plot(grafico, titulo_arquivo, altura=130, largura=150, titulo_plot)

In [ ]:
titulo_plot <- "CRAs (Turmas de 2006.1 a 2014.2)"
titulo_arquivo <- "indicadores_fluxo_sexo"

title_theme <- theme(plot.title=element_text(family='', face='bold', colour='black', size=10))

p1 <- visualizacoes_inep_homem[[1]] + 
    ggtitle("Homens") +
    title_theme +
    plot_layout(guides = 'collect', axes = "collect") +
    plot_annotation(
        tag_levels = "A",
        title = titulo_plot,
        theme = tema + theme(legend.position = "none", plot.title=element_text(family='', face='bold', colour='black', size=16))
      )
p2 <- visualizacoes_inep_mulher[[1]] + 
    ggtitle("Mulheres") +
    title_theme +
    plot_layout(guides = 'collect', axes = "collect") +
    plot_annotation(
        tag_levels = "A",
        title = titulo_plot,
        theme = tema + theme(legend.position = "none", plot.title=element_text(family='', face='bold', colour='black', size=16))
      )

grafico <- (p1 + p2) +
    plot_annotation(
        tag_levels = "A",
        title = titulo_plot,
        theme = tema + theme(legend.position = "bottom", plot.title=element_text(family='', face='bold', colour='black', size=16))
      )

salvar_plot(grafico, titulo_arquivo, altura=180, largura=220, titulo_plot)

## Indicadores de fluxo básico - SEXO - (Turmas de 2006.1 a 2014.2)

## Semestre de evasão por tipo e sexo (Turmas de 2006.1 a 2014.2)

In [ ]:
titulo_arquivo <- "boxplot_evasao_agregado_sexo"
titulo_plot <- "Semestre de evasão por tipo e sexo (Turmas de 2006.1 a 2014.2)"
grafico <- boxplot_agregado[[1]]
tema_modificador <- theme(
        plot.title=element_text(family='', face='bold', colour='black', size=12),
    )

salvar_plot(grafico, titulo_arquivo, altura=160, largura=200, titulo_plot, tema_modificador)

## IC desistentes por semestre SEXO

In [ ]:
titulo_arquivo <- "ic_desistentes_sexo_semestre"
grafico <- ic_evasao_agregado_sexo + scale_x_discrete(name ="Semestre", 
                    limits=c(1,2,3,4,5,6,7,8,9,10,11,12,13))

salvar_plot(grafico, titulo_arquivo, altura=140, largura=180)

## IC concluintes por semestre SEXO

In [ ]:
titulo_arquivo <- "ic_concluintes_sexo_semestre"
grafico <-ic_conclusao_agregado_sexo + scale_x_discrete(name ="Semestre", 
                    limits=c(1,2,3,4,5,6,7,8,9,10,11,12,13))
salvar_plot(grafico, titulo_arquivo, altura=140, largura=180)

## Distribuição dos CRAs

In [ ]:
titulo <- "Distribuição de CRAs por sexo (2006.1 - 2014.2)"

p1 <- boxplot_cras_group_sexo(alunos_raw_cra %>% filter(status == "desistente"), "Agregado", "Desistentes") + 
    theme(show.legend = FALSE)
p2 <- boxplot_cras_group_sexo(alunos_raw_cra %>% filter(status == "graduado"), "Agregado", "Concluintes") + 
    theme(show.legend = FALSE)
    

boxplot_cras_sexo_agregado <- (p1 + p2) +
        plot_annotation(
        tag_levels = "A",
        title = titulo,
        theme = theme_light() + theme(legend.position = "right")
      ) +
      plot_layout(guides = "collect")

ggsave("boxplot_cra_sexo_agregado.png", # the name of the file where it will be save
       plot = boxplot_cras_sexo_agregado, # what plot to save
       height=150, width=220, units="mm")

### Distribuição de CRA por marco

In [ ]:
dist_cras_marco <- plota_junto_4(
    visualizacoes_cra_todos_m1[[4]] + ggtitle(t_m1),
    visualizacoes_cra_todos_m2[[4]] + ggtitle(t_m2),
    visualizacoes_cra_todos_m3[[4]] + ggtitle(t_m3),
    visualizacoes_cra_todos_m4[[4]] + ggtitle(t_m4),
    "Distribuição dos CRAs por marco")

titulo_arquivo <- "dist_cras_marco"
grafico <- dist_cras_marco

salvar_plot(grafico, titulo_arquivo, altura=100, largura=120)

### Distribuição de CRA DESISTENTE por marco

In [ ]:
dist_cras_marco <- plota_junto_4(
    visualizacoes_cra_todos_m1[[5]] + ggtitle(t_m1),
    visualizacoes_cra_todos_m2[[5]] + ggtitle(t_m2),
    visualizacoes_cra_todos_m3[[5]] + ggtitle(t_m3),
    visualizacoes_cra_todos_m4[[5]] + ggtitle(t_m4),
    "Distribuição dos CRAs dos desistentes por marco")

ggsave("dist_cras_desistentes_marco.png", 
       plot = dist_cras_marco, 
       height=100, width=120, units="mm")

img <- magick::image_read(paste('/kaggle/working/', "dist_cras_desistentes_marco.png", sep=""))
    plot(img)

### Distribuição de CRA CONCLUINTE por marco

In [ ]:
dist_cras_marco <- plota_junto_4(
    visualizacoes_cra_todos_m1[[6]] + ggtitle(t_m1),
    visualizacoes_cra_todos_m2[[6]] + ggtitle(t_m2),
    visualizacoes_cra_todos_m3[[6]] + ggtitle(t_m3),
    visualizacoes_cra_todos_m4[[6]] + ggtitle(t_m4),
    "Distribuição dos CRAs dos concluintes por marco")

ggsave("dist_cras_concluintes_marco.png", 
       plot = dist_cras_marco, 
       height=100, width=120, units="mm")

img <- magick::image_read(paste('/kaggle/working/', "dist_cras_concluintes_marco.png", sep=""))
    plot(img)

### Semestre de conclusão por turmas

In [ ]:
regions <- tibble(x1 = -Inf, x2 = +Inf, y1 = 11, y2 = 14)

bp_longitudinal <- gera_boxplot_longitudinal(
    dado_raw %>% filter(graduado == TRUE),
    "Semestre de conclusão por turma"
) + geom_rect(data = regions,
            inherit.aes = FALSE,
            mapping = aes(xmin = x1, xmax = x2,
                          ymin = y1, ymax = y2),
            fill = "dodgerblue2",
            alpha = .2) +
tema_para_exportar +
theme(
    axis.text.x = element_text(size = 8, hjust = 1, angle = 90),
)


ggsave("semestre_de_conclusao_por_turma.png", 
       plot = bp_longitudinal, 
       height=120, width=150, units="mm")

img <- magick::image_read(paste('/kaggle/working/', "semestre_de_conclusao_por_turma.png", sep=""))
    plot(img)

In [ ]:
regions <- tibble(x1 = -Inf, x2 = +Inf, y1 = 3, y2 = 4)

bp_longitudinal <- gera_boxplot_longitudinal(
    dado_raw %>% filter(desistente == TRUE),
    "Semestre de desistência por turma"
) + geom_rect(data = regions,
            inherit.aes = FALSE,
            mapping = aes(xmin = x1, xmax = x2,
                          ymin = y1, ymax = y2),
            fill = "dodgerblue2",
            alpha = .2) +
tema_para_exportar +
theme(
    axis.text.x = element_text(size = 8, hjust = 1, angle = 90),
)

ggsave("semestre_de_desistencia_por_turma.png", 
       plot = bp_longitudinal, 
       height=120, width=150, units="mm")

img <- magick::image_read(paste('/kaggle/working/', "semestre_de_desistencia_por_turma.png", sep=""))
    plot(img)

In [ ]:
regions <- tibble(x1 = -Inf, x2 = +Inf, y1 = 4.7, y2 = 5.4)

grafico <- boxplot_cras_group_marco(alunos_raw_cra, "Todos", "todos") + 
    geom_rect(data = regions,
            inherit.aes = FALSE,
            mapping = aes(xmin = x1, xmax = x2,
                          ymin = y1, ymax = y2),
            fill = "dodgerblue2",
            alpha = .2) +
    tema_para_exportar +
    theme(
        axis.text.x = element_text(size = 8, hjust = 1, angle = 90),
    )

ggsave("bp_cras_marco.png", 
       plot = grafico, 
       height=130, width=130, units="mm")

img <- magick::image_read(paste('/kaggle/working/', "bp_cras_marco.png", sep=""))
    plot(img)

In [ ]:
grafico <- boxplot_cras_group_marco_sexo(
    alunos_raw_cra %>% filter(status == "desistente") %>% filter(sexo != "Todos"),
    "Desistentes") + 
    tema_para_exportar

ggsave("bp_cras_desistentes_sexo.png", 
       plot = grafico, 
       height=130, width=130, units="mm")

img <- magick::image_read(paste('/kaggle/working/', "bp_cras_desistentes_sexo.png", sep=""))
    plot(img)

In [ ]:
grafico <- boxplot_cras_group_marco_sexo(
    alunos_raw_cra %>% filter(status == "graduado") %>% filter(sexo != "Todos"),
    "Concluintes") + 
    tema_para_exportar

ggsave("bp_cras_concluintes_sexo.png", 
       plot = grafico, 
       height=130, width=130, units="mm")

img <- magick::image_read(paste('/kaggle/working/', "bp_cras_concluintes_sexo.png", sep=""))
    plot(img)

In [ ]:
grafico <- conclusao_longitudinal_sexo_marco +
    tema_para_exportar +
    theme(
        axis.text.x = element_text(size = 8, hjust = 1, angle = 45),
    ) 

ggsave("concluintes_sexo_marco.png", 
       plot = grafico, 
       height=100, width=120, units="mm")

img <- magick::image_read(paste('/kaggle/working/', "concluintes_sexo_marco.png", sep=""))
    plot(img)

In [ ]:
grafico <- evasao_longitudinal_sexo_marco +
    tema_para_exportar +
    theme(
        axis.text.x = element_text(size = 8, hjust = 1, angle = 45),
    ) 

ggsave("desistentes_sexo_marco.png", 
       plot = grafico, 
       height=100, width=120, units="mm")

img <- magick::image_read(paste('/kaggle/working/', "desistentes_sexo_marco.png", sep=""))
    plot(img)

In [ ]:
titulo <- "numero_ingressos_sexo"
grafico <- analise_exp_1

salvar_plot(grafico, titulo, altura=140, largura=140)

In [ ]:
titulo <- "proporcao_sexo_ingressantes"
grafico <- analise_exp_2

salvar_plot(grafico, titulo, altura=140, largura=140)

In [ ]:
titulo <- "numero_reingresso_pessoa"
grafico <- analise_exp_3

salvar_plot(grafico, titulo, altura=100, largura=100)

In [ ]:
titulo <- "proporcao_status_e_reingresso_no_curso"
grafico <- analise_exp_4

salvar_plot(grafico, titulo, altura=150, largura=130)

In [ ]:
titulo_arquivo <- "cor_alunos"
grafico <- analise_exp_5
tema_modificador <- theme(axis.text.x = element_text(size = 8, hjust = 1, angle = 45))

salvar_plot(grafico, titulo_arquivo, altura=100, largura=120, tema_modificador=tema_modificador)

In [ ]:
titulo_arquivo <- "prop_cores_alunos"
grafico <- analise_exp_6

salvar_plot(grafico, titulo_arquivo, altura=140, largura=140)

In [ ]:
titulo_arquivo <- "prop_sexo_cor"
grafico <- analise_exp_7

salvar_plot(grafico, titulo_arquivo, altura=100, largura=100)

In [ ]:
titulo_arquivo <- "prop_reingressos_por_periodo"
grafico <- analise_exp_8

salvar_plot(grafico, titulo_arquivo, altura=160, largura=160)

In [ ]:
titulo_arquivo <- "prop_status_periodo"
grafico <- analise_exp_9

salvar_plot(grafico, titulo_arquivo, altura=140, largura=140)